# Day 1 — Pipeline & Sanity Check
   - TQuAD v0.1 yüklendi, schema doğrulandı
   - savasy/bert-base-turkish-squad ile manuel inference test edildi
   - 9/10 semantik doğru, 1/10 [CLS] = no-answer
   - EM yanıltıcı; Gün 2'de F1 eklenecek

In [2]:
# === Cell 1: Environment & version check ===
# Amaç: GPU görünüyor mu, paket versiyonları ne, dataset path doğru mu — bunları sabitle.
# Reproducibility için: notebook'u "Pin to original environment" ile sabitledik,
# yani bu versiyonlar önümüzdeki günlerde değişmeyecek.

import sys, torch, transformers, datasets
import os, subprocess

print("=" * 60)
print("PYTHON & PACKAGE VERSIONS")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")

print("\n" + "=" * 60)
print("GPU CHECK")
print("=" * 60)
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print("\n" + "=" * 60)
print("DATASET PATH (Kaggle Input)")
print("=" * 60)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        full = os.path.join(root, f)
        size_mb = os.path.getsize(full) / 1e6
        print(f"  {full}  ({size_mb:.2f} MB)")

PYTHON & PACKAGE VERSIONS
Python       : 3.12.12
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
Datasets     : 4.8.3

GPU CHECK
CUDA available : True
GPU count      : 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)

DATASET PATH (Kaggle Input)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/BIMAS_faaliyet_2025.pdf  (7.93 MB)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/KTLEV_faaliyet_2025.pdf  (1.77 MB)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/ASELS_faaliyet_2025.pdf  (3.99 MB)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/MPARK_faaliyet_2025.pdf  (5.08 MB)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/CWENE_faaliyet_2025.pdf  (2.82 MB)
  /kaggle/input/datasets/inan20/kap-faaliyet-2025/YEOTK_faaliyet_2025.pdf.pdf  (9.08 MB)
  /kaggle/input/datasets/inan20/tquad-v1/dev-v0.1.json  (0.46 MB)
  /kaggle/input/datasets/inan20/tquad-v1/train-v0.1.json  (3.08 MB)


In [3]:
# === Cell 2: Load TQuAD and validate schema ===
# Amaç: SQuAD-uyumlu mu, kaç örnek var, bir örnek nasıl görünüyor.
# datasets library kullanmıyoruz çünkü TQuAD'in HF Hub upload'ı resmi değil
# ve schema'sı tutarsız — direkt JSON okumak daha güvenli + kontrol bizde.

import json
from pathlib import Path

DATA_DIR  = Path("/kaggle/input/datasets/inan20/tquad-v1")
TRAIN_PATH = DATA_DIR / "train-v0.1.json"
DEV_PATH   = DATA_DIR / "dev-v0.1.json"

def load_squad_format(path):
    """SQuAD v1 formatlı JSON'u flat (question, context, answers) listesine çevir."""
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    examples = []
    for article in raw["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                examples.append({
                    "id":       qa["id"],
                    "question": qa["question"],
                    "context":  context,
                    "answers":  qa["answers"],   # liste: [{"text": ..., "answer_start": ...}, ...]
                    "title":    article["title"],
                })
    return examples, raw.get("version", "?")

train_examples, train_ver = load_squad_format(TRAIN_PATH)
dev_examples,   dev_ver   = load_squad_format(DEV_PATH)

print(f"TQuAD version (train file): {train_ver}")
print(f"Train examples : {len(train_examples)}")
print(f"Dev examples   : {len(dev_examples)}")

# Schema sanity check — ilk örnek tüm beklenen alanlara sahip mi?
sample = train_examples[0]
print("\n--- Sample example ---")
print(f"ID       : {sample['id']}")
print(f"Title    : {sample['title']}")
print(f"Question : {sample['question']}")
print(f"Context  : {sample['context'][:200]}...")
print(f"Answers  : {sample['answers']}")

# Cevap ground-truth doğrulaması: answer_start + len(text) gerçekten context içinde mi?
ans_text  = sample["answers"][0]["text"]
ans_start = int(sample["answers"][0]["answer_start"])
extracted = sample["context"][ans_start : ans_start + len(ans_text)]
print(f"\nGround-truth answer text   : '{ans_text}'")
print(f"Extracted from context     : '{extracted}'")
print(f"Match: {extracted == ans_text}")

TQuAD version (train file): ?
Train examples : 8308
Dev examples   : 892

--- Sample example ---
ID       : 1
Title    : İslamda bilim ve teknik
Question : el-Hasan b. Muhammed el-Vezzan isimli bilgin avrupa’da nasıl tanınmaktadır ?
Context  : İslam dünyasında bilimin 16. yüzyılda hala yüksek seviyede bulunduğunu gösteren çok ilginç bir örneği deskriptif coğrafya ekolünden verebiliriz. Bize bu örneği, Avrupa’da Afrikalı Leo (Leo Africanus) ...
Answers  : [{'answer_start': '171', 'text': 'Afrikalı Leo'}]

Ground-truth answer text   : 'Afrikalı Leo'
Extracted from context     : 'Afrikalı Leo'
Match: True


In [4]:
# === Cell 3: Load sanity-check model + write manual inference ===
# Model: savasy/bert-base-turkish-squad (zaten TQuAD üzerinde fine-tuned)
# Amaç: inference logic'imizin doğru span çıkardığını doğrulamak.
# Gün 2'de fine-tune için BAŞKA model (dbmdz/bert-base-turkish-cased) kullanacağız —
# bu sadece pipeline test'i. Modelin uncased olması sanity check için fark etmez.

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

MODEL_NAME = "savasy/bert-base-turkish-squad"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print("Model loaded.")
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


@torch.no_grad()
def answer_question(question: str, context: str, max_length: int = 384):
    """
    Manuel QA inference:
    1. (question, context) çiftini tokenize et
    2. Modeli çalıştır → start_logits, end_logits
    3. En olası start ve end pozisyonlarını bul
    4. O aralıktaki token'ları decode et → cevap stringi

    Transformers v5'te pipeline("question-answering") kaldırıldığı için manuel yazıyoruz.
    Bu basit versiyon: tek context, no-answer detection yok, multi-span yok.
    Sanity check için yeterli.
    """
    inputs = tokenizer(
        question,
        context,
        max_length=max_length,
        truncation="only_second",   # çok uzun context'i kes, soruyu kesme
        return_tensors="pt",
    ).to(DEVICE)

    outputs = model(**inputs)

    # En yüksek olasılıklı start ve end token indeksleri
    start_idx = outputs.start_logits.argmax().item()
    end_idx   = outputs.end_logits.argmax().item()

    # Korumalar:
    # - end < start ise model emin değil demektir → boş cevap
    # - aralık çok uzunsa muhtemelen yanlış → ilk 30 token ile sınırla
    if end_idx < start_idx or (end_idx - start_idx) > 30:
        return ""

    # Token ID'lerini stringe çevir
    answer_ids = inputs["input_ids"][0, start_idx : end_idx + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    # Confidence skoru — start ve end logit'lerinin softmax sonrası çarpımı
    start_prob = torch.softmax(outputs.start_logits, dim=-1)[0, start_idx].item()
    end_prob   = torch.softmax(outputs.end_logits,   dim=-1)[0, end_idx].item()
    confidence = start_prob * end_prob

    return {"answer": answer, "confidence": confidence,
            "start_idx": start_idx, "end_idx": end_idx}


# Tek örnekle hızlı smoke test
test_q = train_examples[0]["question"]
test_c = train_examples[0]["context"]
test_gt = train_examples[0]["answers"][0]["text"]

result = answer_question(test_q, test_c)
print("\n--- Smoke test ---")
print(f"Q  : {test_q}")
print(f"GT : {test_gt}")
print(f"P  : {result['answer']}")
print(f"Conf: {result['confidence']:.4f}")

Loading savasy/bert-base-turkish-squad on cuda...


config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: savasy/bert-base-turkish-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Model params: 110.0M

--- Smoke test ---
Q  : el-Hasan b. Muhammed el-Vezzan isimli bilgin avrupa’da nasıl tanınmaktadır ?
GT : Afrikalı Leo
P  : Afrikalı Leo
Conf: 0.9999


In [5]:
# === Cell 4: Batch sanity check on 10 dev examples ===
# Amaç: inference fonksiyonu farklı sorularda da çalışıyor mu, edge case var mı?
# Dev set kullanıyoruz (modelin train'de görmediği) — daha gerçekçi sinyal.
# Not: savasy modeli TQuAD train üzerinde fine-tuned, dev'de görmemiş bile olsa
# distribution-içi → yüksek skor beklenir. Asıl iş Gün 4'te finansal eval'de.

import random
random.seed(42)  # reproducible

# Dev'den 10 random örnek seç
sample_size = 10
indices = random.sample(range(len(dev_examples)), sample_size)
samples = [dev_examples[i] for i in indices]

correct_exact = 0
results = []

for i, ex in enumerate(samples, 1):
    gt = ex["answers"][0]["text"]
    res = answer_question(ex["question"], ex["context"])

    # Boş döndüyse (guard tetiklendi)
    if isinstance(res, str) and res == "":
        pred, conf = "", 0.0
    else:
        pred, conf = res["answer"], res["confidence"]

    # Basit normalizasyon: lowercase + strip
    is_exact = pred.strip().lower() == gt.strip().lower()
    if is_exact:
        correct_exact += 1

    results.append({
        "Q": ex["question"][:70],
        "GT": gt,
        "Pred": pred,
        "Conf": f"{conf:.3f}",
        "Match": "✅" if is_exact else "❌",
    })

# Tablo formatında print
print(f"{'#':<3} {'Match':<7} {'Conf':<8} {'GT':<30} {'Pred':<30}")
print("-" * 90)
for i, r in enumerate(results, 1):
    print(f"{i:<3} {r['Match']:<7} {r['Conf']:<8} {r['GT'][:28]:<30} {r['Pred'][:28]:<30}")

print(f"\nExact match: {correct_exact}/{sample_size} ({100*correct_exact/sample_size:.0f}%)")
print("\nNot: bu sadece exact-match. Gerçek QA evaluation F1 (token overlap) kullanır,")
print("Gün 2'de eklenecek. Şimdilik sadece pipeline çalışıyor mu doğrulaması.")

#   Match   Conf     GT                             Pred                          
------------------------------------------------------------------------------------------
1   ❌       0.986    Formula G 2006 Ege Kupası, F                                 
2   ✅       1.000    Siddhanta                      Siddhanta                     
3   ❌       0.917    27 Aralık 2005'te              27 Aralık 2005                
4   ❌       0.834    Hace-i Sultani                 Hace - i Sultani              
5   ❌       0.962    7. yüzyıla                     7. yüzyıla kadar              
6   ❌       0.881    12                             İlk 12                        
7   ❌       0.853    270                            270 öğrenciyle                
8   ✅       0.930    Ursula Sezgin                  Ursula Sezgin                 
9   ✅       1.000    Bostanzade Mehmed Efendi       Bostanzade Mehmed Efendi      
10  ❌       1.000    Nasiriler                      Nasiriler döneminde        

Day 1 — Backlog (rapora ve sonraki günlere taşınacaklar)
🔧 Kod / pipeline TODO (sonraki günlerde eklenecek)

answer_start int cast (Gün 2, fine-tune preprocessing'de)
TQuAD'de answer_start string olarak geliyor ("171"). Tokenizer alignment için int() cast şart, yoksa start_positions hesaplaması patlar.
F1 / EM evaluation fonksiyonu (Gün 2)
SQuAD-style normalized F1 (lowercase + punctuation strip + article removal + whitespace tokenize). HuggingFace evaluate.load("squad") kullanılabilir veya kendimiz yazarız (~30 satır). Türkçe için article removal kısmı atlanabilir.
[CLS] token "no-answer" detection (Gün 4'te lazım olursa)
Şu anki guard sadece end < start ve span uzunluğuna bakıyor. Daha doğrusu: start_logits[CLS] + end_logits[CLS] skorunu en iyi span skoruna karşı eşik olarak kullan. Finansal eval'de "cevabı olmayan soru" sormazsak gerek yok, ama sorarsak gerekli.
Sliding window for long contexts (Gün 4'te lazım olursa)
Şu an truncation="only_second" ile 384 token'dan uzun context kesiliyor. Finansal raporlarda paragraflar uzun olabilir — return_overflowing_tokens=True + stride=128 ile sliding window yazıp her chunk'ta inference + en yüksek confidence'lı cevabı seçmek. ~15 satır eklenti.
Confidence-based filtering (Gün 4 hata analizi için)
Conf < 0.5 olan tahminleri ayrı kategoride incelemek — model "bilmiyorum"u nerede sinyalliyor görmek için.


📄 Rapora ekleme adayları
Yöntem bölümü

"Transformers v5 pipeline kaldırıldı, manuel inference yazdık" notu
Kısa bir paragraf: extractive QA nasıl çalışır (start/end logit head, span argmax, decode). Hocaya "kara kutu kullanmadık" sinyali.
Sanity check için neden farklı model kullandığımızın açıklaması
savasy/bert-base-turkish-squad (uncased) sadece inference logic doğrulaması için; fine-tune'umuz dbmdz/bert-base-turkish-cased üzerinden sıfırdan. Model identity karışıklığı raporda netleşmeli.
EM vs F1 metrik tartışması
"EM annotator'ın span sınırına aşırı duyarlı; semantik doğru cevap bile sınır farkıyla sıfırlanıyor (örn. 'Nasiriler' vs 'Nasiriler döneminde'). F1 birincil metrik olarak seçildi." Bu cümle Gün 1'in en değerli bulgusu.

Bulgular / hata analizi bölümü

TQuAD veri kalite gözlemleri

answer_start string olarak geliyor (schema tutarsızlığı)
Bazı annotation'lar truncated görünüyor (Sample #1: "Formula G 2006 Ege Kupası, F" — cümle ortasında kesilmiş)
Türkçe için bilinen quality varies notu


[CLS] = no-answer davranışı
Model güvenmediği soruda [CLS]'e işaret ediyor → boş string. Standart BERT-QA davranışı, bizim de gözlemlediğimiz. Hata analizi tablosunda ayrı kategori olabilir.
Sanity check tablosu (10 örnek)
Olduğu gibi rapora konabilir — "pipeline'ın çalıştığının kanıtı + EM'in yanıltıcılığının somut örneği" çift işlev.

Trade-off / limitasyonlar bölümü

Sliding window kullanmadık → uzun context'te potansiyel kayıp
Bilinçli trade-off, neden atlandığı + gerekirse nasıl eklenir.
savasy modeli uncased, fine-tune base'imiz cased — bu fark
Sanity check sonuçlarını "modelimizin baseline'ı" olarak yorumlamamak gerektiğinin notu. Gün 2 baseline'ı asıl referans.


📦 Operasyonel TODO

Notebook commit (her gün sonu, "Save & Run All")
GitHub repo'ya notebook export (Gün 5'te toplu)
Coursera sertifikası (paralel iş, dönem sonuna kadar)
HF_TOKEN ayarlama (opsiyonel, hız için — gerek olursa)

In [6]:
# === Cell 4: F1/EM evaluator on dev set with savasy baseline ===
# Amaç:
#   1. evaluate.load("squad") metric'ini kur
#   2. TQuAD answers formatını squad metric formatına çevir
#   3. savasy model ile 892 dev örneğinde F1/EM ölç → baseline rakam
#   4. Bu eval pipeline'ı sonra fine-tune ettiğimiz dbmdz model için tekrar kullanacağız

# --- Install check ---
try:
    import evaluate
    print(f"evaluate already installed: v{evaluate.__version__}")
except ImportError:
    print("evaluate not found, installing...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "evaluate"], check=True)
    import evaluate
    print(f"Installed: v{evaluate.__version__}")

from tqdm.auto import tqdm

# --- Metric yükle ---
# squad metric internet ister (HF hub'dan script çeker). Kaggle'da
# notebook settings → Internet ON olmalı. Kapalıysa açık hata verir.
squad_metric = evaluate.load("squad")
print("squad metric loaded.")

# --- TQuAD → squad metric format ---
# squad metric'in beklediği format:
#   predictions: [{"id": str, "prediction_text": str}, ...]
#   references : [{"id": str, "answers": {"text": [str, ...],
#                                          "answer_start": [int, ...]}}, ...]
# TQuAD'de answers zaten liste-of-dict ama answer_start STRING gelebiliyor
# (Day 1 backlog #1). Burada int'e cast ediyoruz.

def to_reference(example):
    """TQuAD örneğini squad metric reference formatına çevir."""
    texts  = [a["text"] for a in example["answers"]]
    starts = [int(a["answer_start"]) for a in example["answers"]]  # string→int
    return {"id": str(example["id"]),
            "answers": {"text": texts, "answer_start": starts}}

# --- Dev set'te inference + tahminleri biriktir ---
# Trade-off: single-example inference yapıyoruz (Cell 3'teki answer_question).
# 892 örnek × ~80ms ≈ 70sn — kabul edilebilir. Batched inference 5-10x hızlı
# ama collator + padding handling karmaşıklığı katar. Fine-tune evaluation
# için sonra batched yazarız; baseline için single yeterli.

predictions = []
references  = []

for ex in tqdm(dev_examples, desc="Inference on dev set"):
    result = answer_question(ex["question"], ex["context"])
    pred_text = result["answer"] if isinstance(result, dict) else ""  # Cell 3 boş string dönebiliyor
    predictions.append({"id": str(ex["id"]), "prediction_text": pred_text})
    references.append(to_reference(ex))

# --- Metric hesapla ---
results = squad_metric.compute(predictions=predictions, references=references)
print("\n=== savasy/bert-base-turkish-squad on TQuAD dev ===")
print(f"Exact Match : {results['exact_match']:.2f}")
print(f"F1          : {results['f1']:.2f}")

# --- Sanity: 5 örneğe bakalım (doğru, yanlış span, tamamen kaçırılmış) ---
print("\n--- Sample predictions ---")
import random
random.seed(42)
for i in random.sample(range(len(dev_examples)), 5):
    gt   = dev_examples[i]["answers"][0]["text"]
    pred = predictions[i]["prediction_text"]
    mark = "✓" if pred.strip() == gt.strip() else "✗"
    print(f"{mark} Q : {dev_examples[i]['question'][:80]}")
    print(f"   GT: '{gt}'")
    print(f"   P : '{pred}'\n")

evaluate not found, installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
Installed: v0.4.6


squad metric loaded.


Inference on dev set:   0%|          | 0/892 [00:00<?, ?it/s]


=== savasy/bert-base-turkish-squad on TQuAD dev ===
Exact Match : 45.63
F1          : 66.69

--- Sample predictions ---
✗ Q : SAGUAR X5 hangi etkinliklere katıldı?
   GT: 'Formula G 2006 Ege Kupası, Formula G 2006 IstanbulPark, Autoshow 2006'
   P : ''

✓ Q : Hindlilerin oluşturduğu ve İran ekolünden gelen bilginlerin Arapçaya çevirdiği e
   GT: 'Siddhanta'
   P : 'Siddhanta'

✗ Q : Pardus'un ilk kurulabilir sürümü hangi tarihte ağ üzerinden paylaşılmıştır?
   GT: '27 Aralık 2005'te'
   P : '27 Aralık 2005'

✗ Q : Hoca Sadeddin Efendi hangi sıfatla devlet işlerinde etkili olmuştur?
   GT: 'Hace-i Sultani '
   P : 'Hace - i Sultani'

✗ Q : Fuat Sezgin'in tezine göre Buhari'nin sözlü hadisleri kaçıncı yüzyıl öncesine ka
   GT: '7. yüzyıla'
   P : '7. yüzyıla kadar'



In [7]:
# === Cell 5: Preprocessing for fine-tune (sliding window + answer alignment) ===
# Amaç:
#   1. dbmdz/bert-base-turkish-cased tokenizer'ı yükle (FINE-TUNE modeli — savasy değil!)
#   2. (question, context) çiftlerini sliding window ile tokenize et
#      - max_length=384, stride=128 → uzun context'ler birden fazla "feature"a bölünür
#   3. TRAIN için: her feature'da cevabın token-level start/end pozisyonunu bul
#      - Cevap o chunk'ta yoksa → start=end=0 (CLS token) "cevap yok" sinyali
#   4. EVAL için: offset_mapping + example_id sakla (logit→char span dönüşümü için)
#
# Trade-off: train ve eval preprocessing AYRI fonksiyonlar — confusing ama gerekli.
# Train'de offset_mapping atıyoruz (model'e gitmesin), eval'da tutuyoruz (postprocessing'de lazım).

from datasets import Dataset
from transformers import AutoTokenizer

FINETUNE_MODEL = "dbmdz/bert-base-turkish-cased"
MAX_LENGTH = 384
STRIDE     = 128

print(f"Loading tokenizer for {FINETUNE_MODEL}...")
ft_tokenizer = AutoTokenizer.from_pretrained(FINETUNE_MODEL)
# Fast tokenizer şart — sequence_ids() ve offset_mapping fast tokenizer'a özel
assert ft_tokenizer.is_fast, "Fast tokenizer gerekli (offset_mapping için)"
print(f"Tokenizer loaded. Fast: {ft_tokenizer.is_fast}, vocab: {ft_tokenizer.vocab_size}")

# --- TQuAD list → HF Dataset (clean .map() için) ---
# answer_start cast'i burada bir kez yapıyoruz, downstream her yerde int.
def normalize_example(ex):
    return {
        "id":       str(ex["id"]),  # squad metric string istiyor (Cell 4 dersi)
        "question": ex["question"].strip(),  # bazı TQuAD sorularında trailing space var
        "context":  ex["context"],
        "answers":  {
            "text":         [a["text"]              for a in ex["answers"]],
            "answer_start": [int(a["answer_start"]) for a in ex["answers"]],
        },
    }

train_ds = Dataset.from_list([normalize_example(e) for e in train_examples])
dev_ds   = Dataset.from_list([normalize_example(e) for e in dev_examples])
print(f"Train ds: {len(train_ds)}, Dev ds: {len(dev_ds)}")


# --- TRAIN preprocessing ---
def prepare_train_features(examples):
    """
    Her example → 1+ feature (sliding window nedeniyle).
    Çıktı: input_ids, attention_mask, start_positions, end_positions
    """
    tokenized = ft_tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",        # uzun olan context, soruyu kesme
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,  # sliding window aktif
        return_offsets_mapping=True,     # char↔token map
        padding="max_length",
    )

    # overflow_to_sample_mapping: feature_idx → example_idx (orijinal hangi soru bu?)
    sample_map     = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"]   = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_idx   = input_ids.index(ft_tokenizer.cls_token_id)  # genelde 0

        # Bu feature hangi orijinal örneğe ait?
        sample_idx = sample_map[i]
        answers    = examples["answers"][sample_idx]

        # SQuAD/TQuAD'de answer listesi var (multi-annotation); train'de ilkini kullan
        if len(answers["answer_start"]) == 0:
            # No-answer örneği (TQuAD v0.1'de yok ama defensive)
            tokenized["start_positions"].append(cls_idx)
            tokenized["end_positions"].append(cls_idx)
            continue

        start_char = answers["answer_start"][0]
        end_char   = start_char + len(answers["text"][0])

        # Context'in token aralığını bul (question token'larından sonrası)
        sequence_ids = tokenized.sequence_ids(i)
        ctx_start = 0
        while sequence_ids[ctx_start] != 1:
            ctx_start += 1
        ctx_end = len(sequence_ids) - 1
        while sequence_ids[ctx_end] != 1:
            ctx_end -= 1

        # Cevap bu chunk'ın context aralığında mı?
        if (offsets[ctx_start][0] > start_char) or (offsets[ctx_end][1] < end_char):
            # Cevap bu chunk dışında → CLS'e işaret et
            tokenized["start_positions"].append(cls_idx)
            tokenized["end_positions"].append(cls_idx)
        else:
            # Char span'ını token span'ına çevir
            tok_start = ctx_start
            while tok_start <= ctx_end and offsets[tok_start][0] <= start_char:
                tok_start += 1
            tokenized["start_positions"].append(tok_start - 1)

            tok_end = ctx_end
            while tok_end >= ctx_start and offsets[tok_end][1] >= end_char:
                tok_end -= 1
            tokenized["end_positions"].append(tok_end + 1)

    return tokenized


# --- EVAL preprocessing ---
def prepare_validation_features(examples):
    """
    Eval'da start/end_positions HESAPLAMA YOK — modelden logit alıp postprocessing'de
    char span çıkaracağız. Onun için offset_mapping + example_id saklamak şart.
    """
    tokenized = ft_tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        # Question tokenlarının offset'ini None yap — postprocessing'de
        # "context dışı pozisyon mu?" check'i için
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized


# --- Uygula ---
print("\nPreprocessing train set...")
train_features = train_ds.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_ds.column_names,  # raw alanları at, sadece tensor'lar kalsın
    desc="Tokenizing train",
)

print("Preprocessing dev set...")
dev_features = dev_ds.map(
    prepare_validation_features,
    batched=True,
    remove_columns=dev_ds.column_names,
    desc="Tokenizing dev",
)

# --- Sanity ---
print(f"\nTrain examples : {len(train_ds)} → features: {len(train_features)} "
      f"(expansion: {len(train_features)/len(train_ds):.2f}x)")
print(f"Dev   examples : {len(dev_ds)} → features: {len(dev_features)} "
      f"(expansion: {len(dev_features)/len(dev_ds):.2f}x)")

# Kaç feature'da cevap CLS'e işaretlendi (chunk dışı kaldı)? Yüksekse stride az demek.
cls_count = sum(1 for s, e in zip(train_features["start_positions"],
                                   train_features["end_positions"])
                if s == 0 and e == 0)
print(f"Train features with CLS-labeled (answer outside chunk): "
      f"{cls_count}/{len(train_features)} ({100*cls_count/len(train_features):.1f}%)")

# Bir örneği decode edip doğrula
print("\n--- Sanity: decode one train feature's labeled span ---")
idx = 0
s_pos = train_features["start_positions"][idx]
e_pos = train_features["end_positions"][idx]
ids   = train_features["input_ids"][idx][s_pos:e_pos+1]
print(f"Decoded label span: '{ft_tokenizer.decode(ids)}'")
print(f"Original answer   : '{train_examples[0]['answers'][0]['text']}'")

Loading tokenizer for dbmdz/bert-base-turkish-cased...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded. Fast: True, vocab: 32000
Train ds: 8308, Dev ds: 892

Preprocessing train set...


Tokenizing train:   0%|          | 0/8308 [00:00<?, ? examples/s]

Preprocessing dev set...


Tokenizing dev:   0%|          | 0/892 [00:00<?, ? examples/s]


Train examples : 8308 → features: 10878 (expansion: 1.31x)
Dev   examples : 892 → features: 1340 (expansion: 1.50x)
Train features with CLS-labeled (answer outside chunk): 2257/10878 (20.7%)

--- Sanity: decode one train feature's labeled span ---
Decoded label span: 'Afrikalı Leo'
Original answer   : 'Afrikalı Leo'


In [8]:
# === Cell 6a: Postprocessing — logits → text predictions via offset mapping ===
# Amaç:
#   Cell 4'teki manuel decode'un "Hace - i Sultani" gibi boşluk problemi vardı.
#   Çözüm: token decode YERİNE offset_mapping ile orijinal context'ten char span KES.
#   Bu HF'in resmi QA postprocessing pattern'i — Trainer compute_metrics ve final eval için aynı fonksiyon.
#
# Logic:
#   1. Her example birden fazla feature'a bölünmüş olabilir (sliding window)
#   2. Her feature için n_best_size kadar en yüksek start/end logit indeksini seç
#   3. Tüm (start, end) kombinasyonlarını skorla, geçersiz olanları ele:
#      - end < start
#      - span > max_answer_length
#      - offset None (question token'ı, context değil)
#   4. En yüksek skorlu span'ın char aralığını context'ten kes → temiz string
#
# Bu fonksiyon test edilmedi — Cell 6b savasy üzerinde validate edecek.

import collections
import numpy as np

def postprocess_qa_predictions(
    examples,            # raw dataset (id, question, context, answers)
    features,            # tokenized features (input_ids, offset_mapping, example_id)
    all_start_logits,    # np.array, shape (n_features, max_length)
    all_end_logits,      # np.array, shape (n_features, max_length)
    n_best_size: int = 20,
    max_answer_length: int = 50,  # default 30, ama TQuAD'de uzun cevaplar var (Cell 4 SAGUAR örneği)
):
    """
    Returns: dict {example_id: predicted_text}
    """
    # example_id → o example'a ait feature indeksleri
    example_id_to_index = {ex["id"]: i for i, ex in enumerate(examples)}
    features_per_example = collections.defaultdict(list)
    for feat_idx, feat in enumerate(features):
        features_per_example[example_id_to_index[feat["example_id"]]].append(feat_idx)

    predictions = collections.OrderedDict()

    for example_idx, example in enumerate(examples):
        feature_indices = features_per_example[example_idx]
        context = example["context"]
        best_score = -float("inf")
        best_text  = ""

        for feature_idx in feature_indices:
            start_logits = all_start_logits[feature_idx]
            end_logits   = all_end_logits[feature_idx]
            offset_mapping = features[feature_idx]["offset_mapping"]

            # En yüksek n_best start ve end indeksleri (argpartition daha hızlı, ama argsort okunaklı)
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes   = np.argsort(end_logits)  [-1 : -n_best_size - 1 : -1].tolist()

            for start_idx in start_indexes:
                for end_idx in end_indexes:
                    # Geçersiz: out-of-range veya question token (offset None)
                    if (start_idx >= len(offset_mapping)
                        or end_idx >= len(offset_mapping)
                        or offset_mapping[start_idx] is None
                        or offset_mapping[end_idx]   is None):
                        continue
                    # Geçersiz: end < start veya span çok uzun
                    if end_idx < start_idx or end_idx - start_idx + 1 > max_answer_length:
                        continue

                    score = start_logits[start_idx] + end_logits[end_idx]
                    if score > best_score:
                        best_score = score
                        start_char = offset_mapping[start_idx][0]
                        end_char   = offset_mapping[end_idx][1]
                        best_text  = context[start_char:end_char]  # ← raw context'ten kes, decode YOK

        predictions[example["id"]] = best_text

    return predictions


print("postprocess_qa_predictions defined.")
print(f"Config: n_best_size=20, max_answer_length=50")

postprocess_qa_predictions defined.
Config: n_best_size=20, max_answer_length=50


In [9]:
# === Cell 6b: Validate postprocessing on savasy (re-run Cell 4 eval with proper char-span extraction) ===
# Amaç:
#   1. dev_features savasy tokenizer'ı ile DEĞİL, dbmdz tokenizer'ı ile hazırlandı (Cell 5)
#      → savasy'i test etmek için dev set'i savasy tokenizer'ı ile yeniden tokenize etmemiz lazım
#   2. savasy modelini batched inference ile çalıştır → start/end logits matrisi
#   3. postprocess_qa_predictions ile char-span predictions üret
#   4. squad metric ile F1/EM hesapla → Cell 4 ile karşılaştır
#
# Beklenti: F1 ~aynı (66'lar), EM ↑↑ (45 → 55+) çünkü "Hace - i Sultani" tarzı boşluk hatası ortadan kalkacak.
# Eğer EM artmazsa postprocessing'de bug var — fine-tune'a geçmeden önce yakalamak şart.

from torch.utils.data import DataLoader

# --- Dev set'i savasy tokenizer'ıyla yeniden tokenize et ---
# Cell 5'teki prepare_validation_features ft_tokenizer'a (dbmdz) hardcoded — geçici olarak
# savasy tokenizer'ıyla tokenize eden lokal versiyon yazalım.

savasy_tokenizer = tokenizer  # Cell 3'te yüklendi

def prepare_val_features_with(tokenizer_):
    def _fn(examples):
        tokenized = tokenizer_(
            examples["question"],
            examples["context"],
            truncation="only_second",
            max_length=MAX_LENGTH,
            stride=STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )
        sample_map = tokenized.pop("overflow_to_sample_mapping")
        tokenized["example_id"] = []
        for i in range(len(tokenized["input_ids"])):
            sample_idx = sample_map[i]
            tokenized["example_id"].append(examples["id"][sample_idx])
            sequence_ids = tokenized.sequence_ids(i)
            tokenized["offset_mapping"][i] = [
                (o if sequence_ids[k] == 1 else None)
                for k, o in enumerate(tokenized["offset_mapping"][i])
            ]
        return tokenized
    return _fn

print("Tokenizing dev set with savasy tokenizer...")
dev_features_savasy = dev_ds.map(
    prepare_val_features_with(savasy_tokenizer),
    batched=True,
    remove_columns=dev_ds.column_names,
    desc="Tokenizing dev (savasy)",
)
print(f"Dev features (savasy): {len(dev_features_savasy)}")

# --- Batched inference ---
# Cell 4'te tek tek yapıyorduk (70sn), şimdi batched (~15sn). 1340 feature × batch 32.
# Trade-off: model'e tensor verirken offset_mapping ve example_id verilemez — atmamız lazım.

import torch

def run_inference(model_, features_, batch_size=32):
    """Tüm feature'ları batch'le, start/end logits matrislerini topla."""
    model_.eval()
    # offset_mapping ve example_id model input değil — sadece postprocessing için saklanıyor
    model_inputs = features_.remove_columns(["offset_mapping", "example_id"])
    model_inputs.set_format("torch")
    loader = DataLoader(model_inputs, batch_size=batch_size, shuffle=False)

    all_start, all_end = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model_(**batch)
            all_start.append(out.start_logits.cpu().numpy())
            all_end.append(out.end_logits.cpu().numpy())
    return np.concatenate(all_start, axis=0), np.concatenate(all_end, axis=0)

print("\nRunning savasy inference on dev features...")
start_logits, end_logits = run_inference(model, dev_features_savasy)
print(f"Logits shape: start {start_logits.shape}, end {end_logits.shape}")

# --- Postprocess → predictions ---
predictions_dict = postprocess_qa_predictions(
    examples=[normalize_example(e) for e in dev_examples],  # raw examples, normalized
    features=dev_features_savasy,
    all_start_logits=start_logits,
    all_end_logits=end_logits,
)

# --- squad metric format ---
preds_formatted = [{"id": str(eid), "prediction_text": ptext}
                   for eid, ptext in predictions_dict.items()]
refs_formatted  = [{"id": str(ex["id"]),
                    "answers": {
                        "text":         [a["text"]              for a in ex["answers"]],
                        "answer_start": [int(a["answer_start"]) for a in ex["answers"]],
                    }}
                   for ex in dev_examples]

results = squad_metric.compute(predictions=preds_formatted, references=refs_formatted)
print("\n=== savasy with PROPER postprocessing (offset mapping) ===")
print(f"Exact Match : {results['exact_match']:.2f}  (Cell 4: 45.63)")
print(f"F1          : {results['f1']:.2f}  (Cell 4: 66.69)")

# --- Aynı 5 örneği gör → boşluk hatası gerçekten gitti mi? ---
print("\n--- Same samples, new predictions ---")
random.seed(42)
sample_indices = random.sample(range(len(dev_examples)), 5)
for i in sample_indices:
    eid  = dev_examples[i]["id"]
    gt   = dev_examples[i]["answers"][0]["text"]
    pred = predictions_dict[str(eid)]
    mark = "✓" if pred.strip() == gt.strip() else "✗"
    print(f"{mark} Q : {dev_examples[i]['question'][:80]}")
    print(f"   GT: '{gt}'")
    print(f"   P : '{pred}'\n")

Tokenizing dev set with savasy tokenizer...


Tokenizing dev (savasy):   0%|          | 0/892 [00:00<?, ? examples/s]

Dev features (savasy): 1340

Running savasy inference on dev features...


Inference:   0%|          | 0/42 [00:00<?, ?it/s]

Logits shape: start (1340, 384), end (1340, 384)

=== savasy with PROPER postprocessing (offset mapping) ===
Exact Match : 62.78  (Cell 4: 45.63)
F1          : 81.08  (Cell 4: 66.69)

--- Same samples, new predictions ---
✗ Q : SAGUAR X5 hangi etkinliklere katıldı?
   GT: 'Formula G 2006 Ege Kupası, Formula G 2006 IstanbulPark, Autoshow 2006'
   P : 'Formula G 2006 Ege Kupası, Formula G 2006 IstanbulPark, Autoshow'

✓ Q : Hindlilerin oluşturduğu ve İran ekolünden gelen bilginlerin Arapçaya çevirdiği e
   GT: 'Siddhanta'
   P : 'Siddhanta'

✗ Q : Pardus'un ilk kurulabilir sürümü hangi tarihte ağ üzerinden paylaşılmıştır?
   GT: '27 Aralık 2005'te'
   P : '27 Aralık 2005'

✓ Q : Hoca Sadeddin Efendi hangi sıfatla devlet işlerinde etkili olmuştur?
   GT: 'Hace-i Sultani '
   P : 'Hace-i Sultani'

✗ Q : Fuat Sezgin'in tezine göre Buhari'nin sözlü hadisleri kaçıncı yüzyıl öncesine ka
   GT: '7. yüzyıla'
   P : '7. yüzyıla kadar'



In [10]:
# === Cell 7 (fixed): Conditional train, skip if model exists ===
from pathlib import Path

OUTPUT_DIR = "/kaggle/working/ft-bert-tr-tquad"
MODEL_EXISTS = (Path(OUTPUT_DIR) / "model.safetensors").exists()

if MODEL_EXISTS:
    print(f"✓ Model zaten var: {OUTPUT_DIR}")
    print("  Cell 7 (eğitim) atlanıyor → Cell 7b yükleyecek")
else:
    print(f"✗ Model yok, eğitim başlıyor...")
    
    import os, gc
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    
    # Fresh model
    try:
        del ft_model, trainer
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
    
    from transformers import (
        AutoModelForQuestionAnswering,
        TrainingArguments,
        Trainer,
        default_data_collator,
    )
    
    print(f"\nLoading {FINETUNE_MODEL}...")
    ft_model = AutoModelForQuestionAnswering.from_pretrained(FINETUNE_MODEL)
    print(f"Model params: {sum(p.numel() for p in ft_model.parameters()) / 1e6:.1f}M")
    
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=3e-5,
        num_train_epochs=2,
        per_device_train_batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        logging_steps=50,
        report_to="none",
    )
    
    trainer = Trainer(
        model=ft_model,
        args=training_args,
        train_dataset=train_features,
        processing_class=ft_tokenizer,
        data_collator=default_data_collator,
    )
    
    print("Trainer ready (training-only mode).")
    print(f"  Steps/epoch : ~{len(train_features) // (16 * torch.cuda.device_count())}")
    print(f"  Total steps : ~{2 * len(train_features) // (16 * torch.cuda.device_count())}")
    
    train_result = trainer.train()
    trainer.save_model(OUTPUT_DIR)
    ft_tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"\nModel saved to {OUTPUT_DIR}")
    print(f"Final training loss: {train_result.training_loss:.4f}")

✗ Model yok, eğitim başlıyor...
GPU free: 15.00 GB

Loading dbmdz/bert-base-turkish-cased...


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

Model params: 110.0M
Trainer ready (training-only mode).
  Steps/epoch : ~339
  Total steps : ~679


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to /kaggle/working/ft-bert-tr-tquad
Final training loss: 1.6793


In [11]:
# === Cell 7b: Reload trained model from disk (skip retraining) ===
from pathlib import Path
OUTPUT_DIR = "/kaggle/working/ft-bert-tr-tquad"

if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    print(f"✓ Model found at {OUTPUT_DIR}")
    print(f"  Files: {[p.name for p in Path(OUTPUT_DIR).iterdir()]}")
    
    from transformers import AutoModelForQuestionAnswering, AutoTokenizer
    ft_model = AutoModelForQuestionAnswering.from_pretrained(OUTPUT_DIR).to(DEVICE)
    ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    ft_model.eval()
    print("✓ Model + tokenizer loaded from disk (no retraining)")
else:
    print(f"✗ Model NOT found at {OUTPUT_DIR} — need to retrain (Cell 7)")

✓ Model found at /kaggle/working/ft-bert-tr-tquad
  Files: ['training_args.bin', 'tokenizer.json', 'model.safetensors', 'tokenizer_config.json', 'config.json', 'checkpoint-680']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Model + tokenizer loaded from disk (no retraining)


In [12]:
# === Cell 8 — başına ekle ===
# Cell 7 artık reload-from-disk olabildiği için, eski compute_metrics closure'undan
# kalan tanımları burada yeniden kuruyoruz. Defensive.
dev_examples_normalized = [normalize_example(e) for e in dev_examples]
refs_formatted = [{"id": str(ex["id"]),
                   "answers": {
                       "text":         [a["text"]              for a in ex["answers"]],
                       "answer_start": [int(a["answer_start"]) for a in ex["answers"]],
                   }}
                  for ex in dev_examples]
# === Cell 8: Final evaluation on dev set ===
# Trainer'ın per-epoch eval'i çalışmadığı için manual. Cell 6b ile aynı pattern,
# tek fark: model = ft_model, tokenizer = ft_tokenizer (dbmdz, savasy değil).
# dev_features (dbmdz tokenizer ile yapılmış, Cell 5) zaten doğru.

print("Running ft_model inference on dev features (dbmdz tokenizer)...")
ft_model.to(DEVICE)
ft_start_logits, ft_end_logits = run_inference(ft_model, dev_features, batch_size=64)
print(f"Logits shape: {ft_start_logits.shape}")

# Postprocess
ft_predictions = postprocess_qa_predictions(
    examples=dev_examples_normalized,
    features=dev_features,
    all_start_logits=ft_start_logits,
    all_end_logits=ft_end_logits,
)
ft_preds_formatted = [{"id": str(eid), "prediction_text": ptext}
                      for eid, ptext in ft_predictions.items()]

ft_results = squad_metric.compute(predictions=ft_preds_formatted, references=refs_formatted)

print("\n" + "="*60)
print("=== FINAL RESULTS on TQuAD dev set ===")
print("="*60)
print(f"{'Model':<45} {'F1':>7} {'EM':>7}")
print("-"*60)
print(f"{'savasy/bert-base-turkish-squad (baseline)':<45} {81.08:>7.2f} {62.78:>7.2f}")
print(f"{'Our fine-tuned dbmdz/bert-base-turkish':<45} {ft_results['f1']:>7.2f} {ft_results['exact_match']:>7.2f}")
delta_f1 = ft_results['f1'] - 81.08
delta_em = ft_results['exact_match'] - 62.78
print(f"{'Δ':<45} {delta_f1:>+7.2f} {delta_em:>+7.2f}")

# Aynı 5 örneği gör → fine-tune sonrası nasıl?
print("\n--- Same 5 dev samples — predictions ---")
random.seed(42)
for i in random.sample(range(len(dev_examples)), 5):
    eid  = dev_examples[i]["id"]
    gt   = dev_examples[i]["answers"][0]["text"]
    pred = ft_predictions[str(eid)]
    mark = "✓" if pred.strip() == gt.strip() else "✗"
    print(f"{mark} Q : {dev_examples[i]['question'][:80]}")
    print(f"   GT: '{gt}'")
    print(f"   P : '{pred}'\n")

Running ft_model inference on dev features (dbmdz tokenizer)...


Inference:   0%|          | 0/21 [00:00<?, ?it/s]

Logits shape: (1340, 384)

=== FINAL RESULTS on TQuAD dev set ===
Model                                              F1      EM
------------------------------------------------------------
savasy/bert-base-turkish-squad (baseline)       81.08   62.78
Our fine-tuned dbmdz/bert-base-turkish          72.11   50.45
Δ                                               -8.97  -12.33

--- Same 5 dev samples — predictions ---
✗ Q : SAGUAR X5 hangi etkinliklere katıldı?
   GT: 'Formula G 2006 Ege Kupası, Formula G 2006 IstanbulPark, Autoshow 2006'
   P : 'Formula G 2006 Ege Kupası'

✓ Q : Hindlilerin oluşturduğu ve İran ekolünden gelen bilginlerin Arapçaya çevirdiği e
   GT: 'Siddhanta'
   P : 'Siddhanta'

✗ Q : Pardus'un ilk kurulabilir sürümü hangi tarihte ağ üzerinden paylaşılmıştır?
   GT: '27 Aralık 2005'te'
   P : '27 Aralık 2005'

✓ Q : Hoca Sadeddin Efendi hangi sıfatla devlet işlerinde etkili olmuştur?
   GT: 'Hace-i Sultani '
   P : 'Hace-i Sultani'

✗ Q : Fuat Sezgin'in tezine göre Buhari

In [13]:
# === Cell 9: Extract text from KAP annual reports (PyMuPDF) ===
# 6 şirket × yıllık faaliyet raporu 2025 → text + sanity metadata
# Bu Gün 3'ün ana girdisi: eval set Q&A çiftlerini bu metinlerden manuel oluşturacaksın.

import subprocess, sys
try:
    import fitz  # PyMuPDF
except ImportError:
    print("Installing PyMuPDF...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import fitz
print(f"PyMuPDF version: {fitz.__doc__.split(chr(10))[0]}")

from pathlib import Path
import json, re

# 🔧 GÜNCELLE: Kaggle dataset path'ine göre
PDF_DIR = Path("/kaggle/input/datasets/inan20/kap-faaliyet-2025")
OUT_PATH = Path("/kaggle/working/kap_texts.json")

assert PDF_DIR.exists(), f"PDF dir not found: {PDF_DIR}"
pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs:\n  " + "\n  ".join(p.name for p in pdfs))

def extract_pdf(path):
    """Single PDF → text + metadata."""
    doc = fitz.open(path)
    pages_text = []
    image_count = 0
    table_pages = 0  # heuristic: çok kısa satırlar + sayı yoğunluğu

    for page in doc:
        text = page.get_text()
        pages_text.append(text)
        image_count += len(page.get_images())
        # Heuristic table detection: tüm satırların >%50'si <30 karakter ise tablo şüphesi
        lines = [l for l in text.split("\n") if l.strip()]
        if lines and sum(1 for l in lines if len(l) < 30) / len(lines) > 0.5:
            table_pages += 1

    full_text = "\n\n".join(pages_text)
    doc.close()

    return {
        "ticker":         path.stem.split("_")[0],
        "filename":       path.name,
        "n_pages":        len(pages_text),
        "n_chars":        len(full_text),
        "n_images":       image_count,
        "table_heavy_pp": table_pages,  # tablo şüpheli sayfa sayısı
        "pages":          pages_text,   # liste, sayfa bazında — chunking için
        "full_text":      full_text,
    }

# Process all
print("\nExtracting...")
results = {}
for pdf in pdfs:
    print(f"  {pdf.name}...", end=" ")
    r = extract_pdf(pdf)
    results[r["ticker"]] = r
    print(f"{r['n_pages']:>3}p, {r['n_chars']:>7,} chars, "
          f"{r['n_images']:>3} imgs, ~{r['table_heavy_pp']:>2} table pp")

# --- Türkçe karakter sanity check ---
print("\n--- Turkish character integrity check ---")
TR_CHARS = set("ÇĞİıÖŞÜçğıöşü")
for tk, r in results.items():
    sample = r["full_text"][:5000]
    tr_count = sum(1 for c in sample if c in TR_CHARS)
    has_mojibake = any(bad in sample for bad in ["Ã§", "Ã¶", "Ã¼", "ÅŸ", "Ä±"])  # UTF-8 → Latin1 bozulması
    status = "✗ MOJIBAKE" if has_mojibake else ("✓" if tr_count > 20 else "⚠ low TR chars")
    print(f"  {tk}: {tr_count:>3} TR chars in first 5k, {status}")

# --- Sample paragraph: her şirketten 1 sayfadan örnek ---
print("\n--- Sample content (page 3-5 to skip cover) ---")
for tk, r in results.items():
    if r["n_pages"] >= 5:
        page = r["pages"][4]  # 5. sayfa
        # İlk anlamlı paragrafı bul (>100 char)
        paragraphs = [p.strip() for p in page.split("\n\n") if len(p.strip()) > 100]
        snippet = paragraphs[0][:300] if paragraphs else page[:300]
        print(f"\n[{tk}] (p.5)")
        print(f"  {snippet}...")

# --- Persist ---
# pages listesini de saklıyoruz — chunking için lazım olacak
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nSaved to {OUT_PATH} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")

Installing PyMuPDF...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 74.8 MB/s eta 0:00:00
PyMuPDF version: PyMuPDF 1.27.2.3: Python bindings for the MuPDF 1.27.2 library.
Found 6 PDFs:
  ASELS_faaliyet_2025.pdf
  BIMAS_faaliyet_2025.pdf
  CWENE_faaliyet_2025.pdf
  KTLEV_faaliyet_2025.pdf
  MPARK_faaliyet_2025.pdf
  YEOTK_faaliyet_2025.pdf.pdf

Extracting...
  ASELS_faaliyet_2025.pdf...  93p, 229,073 chars,  49 imgs, ~37 table pp
  BIMAS_faaliyet_2025.pdf... 129p, 720,784 chars, 347 imgs, ~55 table pp
  CWENE_faaliyet_2025.pdf...  46p,  95,822 chars,  31 imgs, ~16 table pp
  KTLEV_faaliyet_2025.pdf...  46p, 115,162 chars,  72 imgs, ~19 table pp
  MPARK_faaliyet_2025.pdf... 191p, 626,678 chars, 1043 imgs, ~86 table pp
  YEOTK_faaliyet_2025.pdf.pdf...  83p, 159,634 chars, 101 imgs, ~58 table pp

--- Turkish character integrity check ---
  ASELS: 302 TR chars in first 5k, ✓
  BIMAS: 508 TR chars in first 5k, ✓
  CWENE: 327 TR chars in first 5k, ✓
  KTLEV: 368 TR chars in first 5

In [14]:
# === Cell 10: Diagnostic + targeted fixes (CWENE glyph, MPARK layout) ===
import re

# --- Sorun 1: CWENE 'i' glyph bug. Regex fix uygula. ---
# Sadece ൴ (U+0D34) → i. Başka bozuk karakter var mı sadece CWENE'de mi kontrol et.
def diagnose_chars(text):
    """Türkçe metinde olmaması gereken karakterleri say."""
    unusual = {}
    for c in text:
        # ASCII + Latin-1 supplement + Turkish-specific extras + whitespace + common punct
        if ord(c) > 127 and c not in "ÇĞİıÖŞÜçğıöşü€“”‘’–—•…":
            unusual[c] = unusual.get(c, 0) + 1
    # Top 5 sıra
    return sorted(unusual.items(), key=lambda x: -x[1])[:5]

print("--- Unusual non-ASCII chars per company (top 5) ---")
for tk, r in results.items():
    print(f"  {tk}: {diagnose_chars(r['full_text'])}")

# --- CWENE fix ---
print("\n--- Fixing CWENE i-glyph ---")
cwene_text = results["CWENE"]["full_text"]
before = cwene_text[:300]
cwene_fixed = cwene_text.replace("൴", "i")
after = cwene_fixed[:300]
print(f"BEFORE: {before[:200]}")
print(f"AFTER : {after[:200]}")
results["CWENE"]["full_text"] = cwene_fixed
results["CWENE"]["pages"] = [p.replace("൴", "i") for p in results["CWENE"]["pages"]]

# --- MPARK: blocks mode ile yeniden extract ---
print("\n--- Re-extracting MPARK with blocks mode ---")
mpark_pdf = PDF_DIR / "MPARK_faaliyet_2025.pdf"
doc = fitz.open(mpark_pdf)
mpark_pages_blocks = []
for page in doc:
    # blocks mode: layout-aware, multi-column'u daha iyi handle eder
    blocks = page.get_text("blocks")  # liste of (x0, y0, x1, y1, text, block_no, block_type)
    # y koordinatına göre sırala (yukarıdan aşağı), sonra x (soldan sağa)
    blocks_sorted = sorted(blocks, key=lambda b: (round(b[1] / 10), b[0]))
    page_text = "\n\n".join(b[4].strip() for b in blocks_sorted if b[4].strip())
    mpark_pages_blocks.append(page_text)
doc.close()

# Karşılaştır
print("\n--- MPARK p.5: default vs blocks mode ---")
print(f"DEFAULT (first 300): {results['MPARK']['pages'][4][:300]}")
print(f"BLOCKS  (first 300): {mpark_pages_blocks[4][:300]}")

# Kullanıcı kararı: blocks daha iyiyse swap edelim
# (Şimdilik default'u tutuyorum, çıktıyı gör sonra karar ver)

# --- Geniş sample: her şirketten p.10, p.30, p.50 (varsa) ---
print("\n" + "="*60)
print("--- Wider samples: pages 10, 30, 50 ---")
print("="*60)
for tk, r in results.items():
    print(f"\n[{tk}] ({r['n_pages']} pages total)")
    for p_idx in [9, 29, 49]:
        if p_idx < r["n_pages"]:
            text = r["pages"][p_idx]
            # İlk uzun paragrafı al
            chunks = [c.strip() for c in text.split("\n\n") if len(c.strip()) > 80]
            snippet = chunks[0][:200] if chunks else text[:200].strip()
            print(f"  p.{p_idx+1}: {snippet[:180]}...")

# --- Save updated ---
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\n✓ Updated kap_texts.json saved")

--- Unusual non-ASCII chars per company (top 5) ---
  ASELS: [('â', 80), ('−', 22), ('î', 4), ('ł', 1), ('∆', 1)]
  BIMAS: [('â', 246), ('·', 72), ('²', 17), ('±', 8), ('̈', 7)]
  CWENE: [('൴', 5980), ('৻', 173), ('â', 39), ('\uf0b7', 11), ('î', 4)]
  KTLEV: [('â', 50), ('✓', 30), ('±', 4), ('Â', 3), ('̧', 2)]
  MPARK: [('Õ', 7327), ('ú', 2533), ('\x81', 2359), ('÷', 1999), ('ø', 1616)]
  YEOTK: [('\xa0', 47), ('â', 36), ('ﬁ', 26), ('ﬂ', 11), ('°', 5)]

--- Fixing CWENE i-glyph ---
BEFORE:  
1 
 
 
 
 
 
 


 
2 
 
İçindekiler 
1.ŞİRKET GENEL BİLGİLERİ ............................................................................................... 3 
2. SERMAYE, ORTAKLIK YAPISI, İMTİYAZL
AFTER :  
1 
 
 
 
 
 
 


 
2 
 
İçindekiler 
1.ŞİRKET GENEL BİLGİLERİ ............................................................................................... 3 
2. SERMAYE, ORTAKLIK YAPISI, İMTİYAZL

--- Re-extracting MPARK with blocks mode ---

--- MPARK p.5: default vs blocks mode ---
DEFAUL

import subprocess
result = subprocess.run(["du", "-sh", "/kaggle/working/"], capture_output=True, text=True)
print(result.stdout)
result = subprocess.run(["ls", "-la", "/kaggle/working/ft-bert-tr-tquad/"], capture_output=True, text=True)
print(result.stdout)

In [15]:
# === Cell 11: Drop MPARK + Repair CWENE ===

# --- Adım 1: MPARK'ı results'tan çıkar ---
if "MPARK" in results:
    del results["MPARK"]
    print(f"✓ MPARK dropped. Remaining: {list(results.keys())}")

# --- Adım 2: CWENE'deki tüm şüpheli karakterleri çıkar, context'le birlikte göster ---
import re
cwene_text = results["CWENE"]["full_text"]

# Şüpheli setimiz: ASCII dışı + Türkçe seti dışı + yaygın noktalama dışı
TR_VALID = set("ÇĞİıÖŞÜçğıöşü€“”‘’–—•…\xa0")  # \xa0 NBSP, zararsız
suspicious = {}
for c in cwene_text:
    if ord(c) > 127 and c not in TR_VALID:
        suspicious[c] = suspicious.get(c, 0) + 1

# Sıralı listele + her biri için 2 örnek bağlam
print("--- Suspicious characters in CWENE ---")
for char, count in sorted(suspicious.items(), key=lambda x: -x[1]):
    # Karakterin geçtiği yerlerden 2 örnek bul, 30 char context
    positions = [m.start() for m in re.finditer(re.escape(char), cwene_text)][:2]
    contexts = [cwene_text[max(0,p-25):p+25].replace("\n"," ") for p in positions]
    print(f"\n  '{char}' (U+{ord(char):04X}) × {count}")
    for ctx in contexts:
        # Bozuk karakteri [X] ile vurgula
        marked = ctx.replace(char, f"[{char}]")
        print(f"    ...{marked}...")

✓ MPARK dropped. Remaining: ['ASELS', 'BIMAS', 'CWENE', 'KTLEV', 'YEOTK']
--- Suspicious characters in CWENE ---

  '৻' (U+09FB) × 173
    ...nmış ve uygulanmıştır.  R[৻]sk[৻]n Erken  Saptanması K...
    ...ş ve uygulanmıştır.  R[৻]sk[৻]n Erken  Saptanması Kom[৻]...

  'â' (U+00E2) × 39
    ...Enerji, kurucu ortak ve h[â]kim hissedar Tarzan Tarı...
    ...alanda etkin kullanım imk[â]nı sunmaktadır.  Grafen ...

  '' (U+F0B7) × 11
    ...ekilde oluşturulmuştur:  []  Tarzan Tarık Sarvan – ...
    ... Yönetim Kurulu Başkanı  []  Volkan Yılmaz – Yöneti...

  'î' (U+00EE) × 4
    ...tarihli 31797 sayılı Resm[î] Gazete’de yayımlanan “Ş...
    ... Şubat 2022  tarihli Resm[î] Gazete’de yayımlanan Cu...

  '²' (U+00B2) × 3
    ...oplam yaklaşık  274.631 m[²] alana sahip yedi farklı...
    ...apta yıllık 10,7 Milyon m[²] üretim kapasitesiyle fa...

  '°' (U+00B0) × 1
    ...cre ile  backsheet’in 180[°]C sıcaklık altında birbi...


In [16]:
# === Cell 12: Apply CWENE replacements + verify ===

# Sadece kanıtlanmış bozuk karakterler
REPLACEMENTS = {
    "৻":      "i",   # U+09FB, 173 occurrences, 'Risk', 'Komite'
    "\uf0b7": "•",   # PUA bullet, 11 occurrences, bullet point
    # â, î, ², ° dokunulmadı — gerçek Türkçe/teknik karakterler
}

def apply_replacements(text, table):
    for bad, good in table.items():
        text = text.replace(bad, good)
    return text

# Tüm CWENE içeriğine uygula (full_text + pages)
results["CWENE"]["full_text"] = apply_replacements(results["CWENE"]["full_text"], REPLACEMENTS)
results["CWENE"]["pages"] = [apply_replacements(p, REPLACEMENTS) for p in results["CWENE"]["pages"]]

# --- Doğrulama: tekrar şüpheli karakter taraması ---
TR_VALID = set("ÇĞİıÖŞÜçğıöşü€“”‘’–—•…\xa0âîÂÎûÛôÔ²°³¼½¾©®™§")  # genişletilmiş valid set
remaining = {}
for c in results["CWENE"]["full_text"]:
    if ord(c) > 127 and c not in TR_VALID:
        remaining[c] = remaining.get(c, 0) + 1

if remaining:
    print(f"⚠ Hâlâ {sum(remaining.values())} şüpheli karakter var:")
    for c, n in sorted(remaining.items(), key=lambda x: -x[1])[:10]:
        print(f"  '{c}' (U+{ord(c):04X}) × {n}")
else:
    print("✓ CWENE temizlendi, şüpheli karakter kalmadı")

# --- "Risk", "Komite" gibi sözcükler artık düzgün mü? ---
print("\n--- Sample fixed phrases ---")
text = results["CWENE"]["full_text"]
for keyword in ["Risk", "Komite", "Ticaret", "Şirket"]:
    # Kelimenin geçtiği ilk yeri bul
    idx = text.find(keyword)
    if idx >= 0:
        print(f"  '{keyword}': ...{text[max(0,idx-20):idx+30]}...")

# --- Save updated ---
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\n✓ kap_texts.json updated ({OUT_PATH.stat().st_size/1e6:.1f} MB)")
print(f"  Companies: {list(results.keys())}")

✓ CWENE temizlendi, şüpheli karakter kalmadı

--- Sample fixed phrases ---
  'Risk': ...yet göstermektedir. Riskin Erken Saptanması Komite...
  'Komite': ...Kurulu Bünyesindeki Komiteler 
Yönetim Kurulu büny...
  'Ticaret': ...aliyet Raporu, Türk Ticaret Kanunu (“TTK”) ve Serm...
  'Şirket': ... GENEL BİLGİLERİ 
 
Şirket Profili ve Genel Bakış ...

✓ kap_texts.json updated (3.0 MB)
  Companies: ['ASELS', 'BIMAS', 'CWENE', 'KTLEV', 'YEOTK']


In [17]:
import json

with open('/kaggle/working/kap_texts.json', 'r', encoding='utf-8') as f:
    kap = json.load(f)

# Yapıyı gör
print("Şirketler:", list(kap.keys()))
print("\nASELS keys:", list(kap['ASELS'].keys()))
print("\nfull_text uzunluk:", len(kap['ASELS']['full_text']), "karakter")
print("\nİlk 500 karakter:")
print(kap['ASELS']['full_text'][:500])

Şirketler: ['ASELS', 'BIMAS', 'CWENE', 'KTLEV', 'YEOTK']

ASELS keys: ['ticker', 'filename', 'n_pages', 'n_chars', 'n_images', 'table_heavy_pp', 'pages', 'full_text']

full_text uzunluk: 229073 karakter

İlk 500 karakter:
0 
 
 
 
31 Aralık 2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Yönetim Kurulu  
Faaliyet Raporu 






 
 
Sayfa No 
Sayfa No 
 
 
 
 
 
 
 
İçindekiler 
 
2    Vizyon-Misyon 
3    Hazırlanma Esasları 
3    Şirket Bilgileri ve Organizasyon Yapısı     
5    Şirketin Ortaklık Yapısı     
8    Şirket’in Performansını Etkileyen Ana Etmenler, 
Uyguladığı Yatırım ve Temettü Politikası 
9    Kurumsal Risk Yönetimi 
11  Şirketin Finansal Bilgileri 
15    Şirket Faaliyetlerini


In [18]:
import json

context = kap['ASELS']['full_text']

# (soru, span, anchor) — anchor: span'in çevresinden unique parça
qa_pairs = [
    ("ASELSAN'ın Yönetim Kurulu Başkanı kimdir?",
     "Ercümend ARVAS", "Ercümend ARVAS"),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla hasılatı ne kadardır?",
     "180,4 Milyar TL", "180,4 Milyar TL olarak gerçekleşmiş"),
    
    ("ASELSAN'da kaç adet Sektör Başkanlığı bulunmaktadır?",
     "altı", "altı ayrı Sektör Başkanlığı"),
    
    ("ASELSAN'ın yönetim kurulu 31 Aralık 2025 itibarıyla kaç kişiden oluşur?",
     "9", "toplam 9 üyeden oluşur"),
    
    ("ASELSAN'ın Genel Müdürü kimdir?",
     "Ahmet AKYOL", "Ahmet AKYOL"),
    
    ("ASELSAN'ın Defense News Top 100'de kaçıncıdır?",
     "43", "2024 sıralamasında 43. sırada"),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla çıkarılmış sermayesi kaç TL'dir?",
     "4.560.000.000", "çıkarılmış sermayesi 4.560.000.000"),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla ihracatının hasılata oranı kaçtır?",
     "%15,3", "ihracatın toplam hasılata oranı %15,3"),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla ortalama personel sayısı kaçtır?",
     "14.143", "ortalama personel sayısı 14.143"),
    
    ("TSKGV'nin ASELSAN'daki pay oranı yüzde kaçtır?",
     "%74,2", "%74,2"),
    
    ("ASELSAN'ın tam konsolidasyona tabi bağlı ortaklıkları hangileridir?",
     "ASELSANNET, ASELSAN Bakü, ASELSAN Sivas Hassas Optik, ASELSAN Global, MKR-IC, ASELSAN Malezya, ASELSAN Konya, BİTES, ASELSAN Ukrayna, ASELSAN Latin Amerika, ASELSAN BAE, ASELSAN MUSCAT, ASELSAN Filipinler, ASELSAN Malatya ve ASELSAN Gaziantep",
     "Tam Konsolidasyona tabi şirketler (ASELSANNET"),
    
    ("TSKGV'nin ASELSAN'daki payının nominal değeri kaç Bin TL'dir?",
     "3.383.302", "(TSKGV)"),
]

# Offset hesaplama + doğrulama
qas = []
for i, (q, span, anchor) in enumerate(qa_pairs):
    anchor_pos = context.find(anchor)
    if anchor_pos == -1:
        print(f"❌ Anchor bulunamadı: Q{i+1}: '{anchor[:50]}...'")
        continue
    
    # Anchor'dan sonra span'i bul
    span_pos = context.find(span, anchor_pos)
    if span_pos == -1:
        print(f"❌ Span bulunamadı: Q{i+1}: '{span[:50]}...'")
        continue
    
    # Doğrulama: o offsetteki text gerçekten span mi?
    extracted = context[span_pos : span_pos + len(span)]
    if extracted != span:
        print(f"❌ Mismatch Q{i+1}: '{extracted[:50]}' != '{span[:50]}'")
        continue
    
    qas.append({
        "id": f"asels_{i+1:03d}",
        "question": q,
        "answers": {
            "text": [span],
            "answer_start": [span_pos]
        }
    })
    print(f"✅ Q{i+1}: offset={span_pos}, span='{span[:40]}...'")

print(f"\nToplam: {len(qas)}/{len(qa_pairs)} soru başarılı")

✅ Q1: offset=11221, span='Ercümend ARVAS...'
❌ Anchor bulunamadı: Q2: '180,4 Milyar TL olarak gerçekleşmiş...'
✅ Q3: offset=5289, span='altı...'
✅ Q4: offset=10782, span='9...'
✅ Q5: offset=12559, span='Ahmet AKYOL...'
❌ Anchor bulunamadı: Q6: '2024 sıralamasında 43. sırada...'
✅ Q7: offset=9520, span='4.560.000.000...'
❌ Anchor bulunamadı: Q8: 'ihracatın toplam hasılata oranı %15,3...'
✅ Q9: offset=51996, span='14.143...'
✅ Q10: offset=10443, span='%74,2...'
❌ Span bulunamadı: Q11: 'ASELSANNET, ASELSAN Bakü, ASELSAN Sivas Hassas Opt...'
✅ Q12: offset=10325, span='3.383.302...'

Toplam: 8/12 soru başarılı


In [19]:
import re

# Whitespace-tolerant arama: tüm boşlukları \s+ ile aç
def find_flex(context, anchor, start=0):
    pattern = re.sub(r'\s+', r'\\s+', re.escape(anchor))
    m = re.search(pattern, context[start:])
    if m:
        return start + m.start(), context[start + m.start() : start + m.end()]
    return -1, None

# Önce başarısızları debug edelim
failures = [
    ("Q2", "180,4 Milyar TL", "180,4 Milyar TL olarak"),
    ("Q6", "43", "2024 sıralamasında 43"),
    ("Q8", "%15,3", "hasılata oranı %15,3"),
    ("Q11", "ASELSANNET, ASELSAN Bakü", "Tam Konsolidasyona tabi şirketler"),  # önce başlangıcı bul
]

for qid, span, anchor in failures:
    pos, actual = find_flex(context, anchor)
    if pos == -1:
        print(f"❌ {qid}: anchor hâlâ bulunamadı")
    else:
        # Çevresini göster (önceki/sonraki 100 char)
        snippet = context[pos : pos+200].replace('\n', '⏎')
        print(f"✅ {qid} anchor pos={pos}")
        print(f"   Çevre: {snippet[:200]}")
        print()

❌ Q2: anchor hâlâ bulunamadı
❌ Q6: anchor hâlâ bulunamadı
❌ Q8: anchor hâlâ bulunamadı
❌ Q11: anchor hâlâ bulunamadı


In [20]:
# Her span için unique ve kısa bir parça arayalım, sonra çevresini RAW olarak gösterelim
targets = [
    ("Q2-hasılat", "180,4"),
    ("Q6-defense", "Defense News"),
    ("Q8-ihracat", "15,3"),
    ("Q11-konsol", "ASELSANNET"),
]

for label, needle in targets:
    pos = context.find(needle)
    if pos == -1:
        print(f"❌ {label}: '{needle}' metinde HİÇ yok!")
        continue
    
    # 150 char öncesi + 250 char sonrası, repr ile (gizli karakterleri görmek için)
    start = max(0, pos - 150)
    end = pos + 250
    snippet = context[start:end]
    print(f"=== {label} | pos={pos} ===")
    print(repr(snippet))  # repr → \n, \xa0, \u200b vb. görünür
    print()

=== Q2-hasılat | pos=44415 ===
've \npazarlama faaliyetlerinin yürütülmesine devam \nedilmektedir.  \n31 Aralık 2025 sonu itibarıyla hasılat geçen yılın \naynı dönemine göre %15 artışla 180,4 Milyar TL \nolarak gerçekleşmiş olup, ihracatın toplam \nhasılata oranı %15,3 olarak gerçekleşmiştir.\n\n\n \n15 \n \ne) Finansal Rasyolar \nGrup’un Sermaye Piyasası Kurulu Seri II, 14.1 Sayılı “Sermaye Piyasasında Finansal Raporlamaya İlişkin \nEsaslar '

=== Q6-defense | pos=6701 ===
'be ve ofislerine ilişkin iletişim bilgileri ile internet \nsitesinin adresi aşağıda tabloda belirtilmiştir: \n \n \n \nGüncel\nBir Önceki\nGüncel\nBir Önceki\nDefense News Top 100\n2025\n43\n42\nAr-Ge Top 100 Araştırması\n2024\n2\n2\nİSO 500 Büyük Sanayi Şirketi\n2024\n17\n19\nEn Gözde 50 Şirket \n(Üniversiteler arası)\n2025\n5\n5\nEn Büyük 500 Şirket\n2024\n26\n31\nİdeal İşverenler \n(Mühendislik ve Bilgi Teknolojileri)\n100 Fi'

=== Q8-ihracat | pos=44492 ===
'2025 sonu itibarıyla hasılat geçen yılın \naynı dönem

In [22]:
import re

def find_flex(ctx, clean_span, start=0):
    """clean_span'i ctx[start:] içinde flexible whitespace ile arar."""
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

# Bilinen pozisyonları kullanarak özel çözümler
recovery = [
    # (qid, clean_span, anchor_to_locate_region)
    ("Q2",  "180,4 Milyar TL", "180,4"),
    ("Q6",  "43",              "Defense News Top 100"),  # next 43 after Defense News
    ("Q8",  "%15,3",           "hasılata oranı"),
    ("Q11", None,              "Tam Konsolidasyona tabi şirketler"),  # special: span = bracket içi
]

recovered = {}
for qid, clean_span, anchor in recovery:
    apos = context.find(anchor)
    if apos == -1:
        print(f"❌ {qid}: anchor bulunamadı")
        continue
    
    if qid == "Q11":
        # Özel: "ASELSANNET"ten ilk ")"e kadar her şeyi al (newline'lar dahil)
        span_start = context.find("ASELSANNET", apos)
        span_end = context.find(")", span_start)
        actual = context[span_start:span_end]
        recovered[qid] = (span_start, actual)
        print(f"✅ {qid}: pos={span_start}, length={len(actual)}")
        print(f"   İlk 80 char (repr): {repr(actual[:80])}")
        print(f"   Son 50 char (repr): {repr(actual[-50:])}")
    else:
        spos, actual = find_flex(context, clean_span, start=apos)
        if spos == -1:
            print(f"❌ {qid}: span yine bulunamadı")
            continue
        recovered[qid] = (spos, actual)
        # Sanity: context[spos:spos+len(actual)] == actual mı?
        check = context[spos:spos+len(actual)] == actual
        print(f"{'✅' if check else '⚠️'} {qid}: pos={spos}, actual={repr(actual)}")

✅ Q2: pos=44415, actual='180,4 Milyar TL'
✅ Q6: pos=6727, actual='43'
✅ Q8: pos=44491, actual='%15,3'
✅ Q11: pos=17315, length=252
   İlk 80 char (repr): 'ASELSANNET, \nASELSAN Bakü, ASELSAN Sivas Hassas Optik, \nASELSAN Global, MKR-IC, '
   Son 50 char (repr): ' Filipinler, ASELSAN Malatya ve \nASELSAN Gaziantep'


In [23]:
import json
import re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

context = kap['ASELS']['full_text']

# (question, clean_span, anchor, special_handler)
qa_specs = [
    ("ASELSAN'ın Yönetim Kurulu Başkanı kimdir?",
     "Ercümend ARVAS", "Ercümend ARVAS", None),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla hasılatı ne kadardır?",
     "180,4 Milyar TL", "180,4", None),
    
    ("ASELSAN'da kaç adet Sektör Başkanlığı bulunmaktadır?",
     "altı", "altı ayrı Sektör Başkanlığı", None),
    
    ("ASELSAN'ın yönetim kurulu 31 Aralık 2025 itibarıyla kaç kişiden oluşur?",
     "9", "toplam 9 üyeden oluşur", None),
    
    ("ASELSAN'ın Genel Müdürü kimdir?",
     "Ahmet AKYOL", "Ahmet AKYOL", None),
    
    ("ASELSAN'ın Defense News Top 100'de kaçıncıdır?",
     "43", "Defense News Top 100", None),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla çıkarılmış sermayesi kaç TL'dir?",
     "4.560.000.000", "çıkarılmış sermayesi 4.560.000.000", None),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla ihracatının hasılata oranı kaçtır?",
     "%15,3", "hasılata oranı", None),
    
    ("ASELSAN'ın 31 Aralık 2025 itibarıyla ortalama personel sayısı kaçtır?",
     "14.143", "ortalama personel sayısı 14.143", None),
    
    ("TSKGV'nin ASELSAN'daki pay oranı yüzde kaçtır?",
     "%74,2", "%74,2", None),
    
    ("ASELSAN'ın tam konsolidasyona tabi bağlı ortaklıkları hangileridir?",
     None, "Tam Konsolidasyona tabi şirketler", "q11_special"),
    
    ("TSKGV'nin ASELSAN'daki payının nominal değeri kaç Bin TL'dir?",
     "3.383.302", "(TSKGV)", None),
]

qas = []
errors = []
for i, (q, clean_span, anchor, special) in enumerate(qa_specs):
    apos = context.find(anchor)
    if apos == -1:
        errors.append(f"Q{i+1}: anchor bulunamadı")
        continue
    
    if special == "q11_special":
        span_start = context.find("ASELSANNET", apos)
        span_end = context.find(")", span_start)
        actual = context[span_start:span_end]
        spos = span_start
    else:
        spos, actual = find_flex(context, clean_span, start=apos)
        if spos == -1:
            errors.append(f"Q{i+1}: span bulunamadı ({clean_span})")
            continue
    
    # Sanity: offset doğru mu?
    assert context[spos:spos+len(actual)] == actual, f"Q{i+1} offset mismatch!"
    
    qas.append({
        "id": f"asels_{i+1:03d}",
        "question": q,
        "answers": {
            "text": [actual],
            "answer_start": [spos]
        }
    })

print(f"✅ {len(qas)}/12 soru hazır")
if errors:
    print("Hatalar:", errors)

# SQuAD-style record
asels_record = {
    "title": "ASELSAN_2025_Faaliyet_Raporu",
    "context": context,
    "qas": qas
}

# Kaydet
with open('/kaggle/working/kap_eval_asels.json', 'w', encoding='utf-8') as f:
    json.dump(asels_record, f, ensure_ascii=False, indent=2)

print(f"\n💾 Kaydedildi: kap_eval_asels.json")
print(f"   Context: {len(context):,} karakter")
print(f"   Soru sayısı: {len(qas)}")
print(f"\nÖrnek soru (Q1):")
print(json.dumps(qas[0], ensure_ascii=False, indent=2))

✅ 12/12 soru hazır

💾 Kaydedildi: kap_eval_asels.json
   Context: 229,073 karakter
   Soru sayısı: 12

Örnek soru (Q1):
{
  "id": "asels_001",
  "question": "ASELSAN'ın Yönetim Kurulu Başkanı kimdir?",
  "answers": {
    "text": [
      "Ercümend ARVAS"
    ],
    "answer_start": [
      11221
    ]
  }
}


In [24]:
import json

# JSON'u yeniden yükle (kernel kapandıysa diye)
with open('/kaggle/working/kap_texts.json', 'r', encoding='utf-8') as f:
    kap = json.load(f)

# BIMAS keşif
text = kap['BIMAS']['full_text']
print(f"Toplam: {len(text):,} karakter")
print(f"Sayfa sayısı: {kap['BIMAS'].get('n_pages', 'N/A')}")
print(f"\nİlk 500 karakter:")
print(text[:500])
print(f"\n{'='*60}")
print("Çeyrek başlangıçları:")
chunk_size = len(text) // 4
for i in range(4):
    start = i * chunk_size
    print(f"\n=== ÇEYREK {i+1} (karakter {start:,}) ===")
    print(text[start:start+300])

Toplam: 720,784 karakter
Sayfa sayısı: 129

İlk 500 karakter:
2050 Net Sıfır Emisyon hedefimiz için sürdürülebilir büyümeyi  rehber 
ediniyoruz.  Doğayla uyumlu bir gelecek inşa etmek için kararlılıkla ilerliyoruz. 
2025 Entegre Faaliyet Raporu
30 yıldır yarını
düşünerek büyüyoruz!


İçindekiler
Giriş
12	
Rapor Hakkında
14	
Yönetim Kurulu Başkanı Mesajı
16	
2025 Yılı Sürdürülebilirlik 
Görünümü
18	
2025 Yılı Yönetim Değerlendirmesi 
ve Performans Analizi
Bir Bakışta BİM
20	
BİM Hakkında
26	
Hizmet Felsefemiz
28	
Faaliyet Coğrafyamız
30	
Kilometre Taşlarımı

Çeyrek başlangıçları:

=== ÇEYREK 1 (karakter 0) ===
2050 Net Sıfır Emisyon hedefimiz için sürdürülebilir büyümeyi  rehber 
ediniyoruz.  Doğayla uyumlu bir gelecek inşa etmek için kararlılıkla ilerliyoruz. 
2025 Entegre Faaliyet Raporu
30 yıldır yarını
düşünerek büyüyoruz!


İçindekiler
Giriş
12	
Rapor Hakkında
14	
Yönetim Kurulu Başkanı Mesajı
16	
20

=== ÇEYREK 2 (karakter 180,196) ===
sunmak amacıyla hayata 
geçirdiğimiz iş sağlı

In [25]:
print("ilk 90.000♦♦♦♦♦♦♦🔴🔴🔴🔵🔵🔵🔵🟢🟢🟢🟢🟢🟢")
print(text[:90000])

print("2. 90.000🟢🟢🟢🔵🔵🔵🟣🟣🟣🟠🟠🟠🔴🔴🔴")
print(text[90000:180000])

ilk 90.000♦♦♦♦♦♦♦🔴🔴🔴🔵🔵🔵🔵🟢🟢🟢🟢🟢🟢
2050 Net Sıfır Emisyon hedefimiz için sürdürülebilir büyümeyi  rehber 
ediniyoruz.  Doğayla uyumlu bir gelecek inşa etmek için kararlılıkla ilerliyoruz. 
2025 Entegre Faaliyet Raporu
30 yıldır yarını
düşünerek büyüyoruz!


İçindekiler
Giriş
12	
Rapor Hakkında
14	
Yönetim Kurulu Başkanı Mesajı
16	
2025 Yılı Sürdürülebilirlik 
Görünümü
18	
2025 Yılı Yönetim Değerlendirmesi 
ve Performans Analizi
Bir Bakışta BİM
20	
BİM Hakkında
26	
Hizmet Felsefemiz
28	
Faaliyet Coğrafyamız
30	
Kilometre Taşlarımız
32	
2025 Yılı Ödül ve Başarılarımız
34	
Başlıca Finansal ve Operasyonel 
Göstergelerimiz
36	
Geleceğe Yönelik Beklentiler/
Gerçekleşmeler
37	
Bağlı Ortaklıklarımız ve 
İştiraklerimiz
2025 Yılı Gelişmeleri
38	
Dünyada Perakendecilik Sektörü
40	 Türkiye’de Perakendecilik Sektörü
41	
Sürdürülebilirlik Ekosistemindeki 
Gelişmeler
Sürdürülebilirlik Yaklaşımı
44	
Sürdürülebilirlik Yönetişimi ve 
Organizasyonu
49	
Paydaş Haritası ve Paydaşlarla 
İletişim
51	
Önceliklend

In [26]:
import json
import re

def find_flex(ctx, clean_span, start=0):
    """Whitespace-tolerant arama."""
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

# JSON tekrar yükle (kernel kapandıysa)
with open('/kaggle/working/kap_texts.json', 'r', encoding='utf-8') as f:
    kap = json.load(f)

context = kap['BIMAS']['full_text']

# (question, clean_span, anchor)
# Anchor = span'in çevresindeki unique text → ondan sonra span'i bul
qa_specs = [
    ("Fas'taki ilk BİM mağazası hangi şehirde açılmıştır?",
     "Kazablanka", "ilk mağazamızı Kazablanka"),
    
    ("BİM hangi yurt dışı ülkelerde faaliyet göstermektedir?",
     "Mısır ve Fas", "Türkiye, Mısır ve Fas"),
    
    ("BİM'in 31 Aralık 2025 itibarıyla net satışları kaç TL'dir?",
     "721,1 Milyar TL net satış", "721,1 Milyar TL net satış"),
    
    ("BİM'in 31 Aralık 2025 itibarıyla toplam mağaza sayısı kaçtır?",
     "14.473", "14.473 mağaza"),
    
    ("BİM'in Yönetim Kurulu Başkanı kimdir?",
     "Mahmud Muhammed Topbaş", "Mahmud Muhammed Topbaş"),
    
    ("BİM'in hızlı tüketim ürünlerinde ciro pazar payı kaçtır?",
     "%18,1", "ciro pazar payı"),
    
    ("BİM'in 31 Aralık 2025 itibarıyla toplam GES kurulu gücü nedir?",
     "96 MW", "kurulu güç 96 MW"),
    
    ("BİM'in 31 Aralık 2025 itibarıyla toplam çalışan sayısı kaçtır?",
     "101.663", "Toplam çalışan sayısı: 101.663"),
    
    ("BİM'in Es Global fabrikası hangi şehirde bulunur?",
     "Eskişehir", "Eskişehir'de yer alan fabrika"),
    
    ("BİM'in 31 Aralık 2025 itibarıyla tedarikçi sayısı kaçtır?",
     "1.434", "Toplam tedarikçi sayısı"),
    
    ("BİM'in Türkiye'de ilk kendi markalı ürünü nedir?",
     "Dost Süt", "ilk kendi markası olan"),
    
    ("FİLE'nin kendi markaları hangileridir?",
     "Harras (Gıda), Actisoft (Temizlik), Daycare (Kişisel Bakım)",
     "FİLE Kendi Markaları"),
]

qas = []
errors = []
for i, (q, clean_span, anchor) in enumerate(qa_specs):
    apos = context.find(anchor)
    if apos == -1:
        # Anchor whitespace-flex ile bulunabilir mi?
        apos, _ = find_flex(context, anchor)
        if apos == -1:
            errors.append(f"Q{i+1}: anchor bulunamadı → '{anchor}'")
            continue
    
    spos, actual = find_flex(context, clean_span, start=apos)
    if spos == -1:
        errors.append(f"Q{i+1}: span bulunamadı → '{clean_span[:50]}'")
        continue
    
    # Sanity
    assert context[spos:spos+len(actual)] == actual, f"Q{i+1} offset mismatch!"
    
    qas.append({
        "id": f"bimas_{i+1:03d}",
        "question": q,
        "answers": {
            "text": [actual],
            "answer_start": [spos]
        }
    })
    print(f"✅ Q{i+1}: pos={spos}, span={repr(actual[:60])}")

print(f"\n{'='*50}")
print(f"Toplam: {len(qas)}/{len(qa_specs)} soru başarılı")
if errors:
    print("\nHatalar:")
    for e in errors:
        print(f"  {e}")

# SQuAD-style record
bimas_record = {
    "title": "BIM_2025_Entegre_Faaliyet_Raporu",
    "context": context,
    "qas": qas
}

with open('/kaggle/working/kap_eval_bimas.json', 'w', encoding='utf-8') as f:
    json.dump(bimas_record, f, ensure_ascii=False, indent=2)

print(f"\n💾 Kaydedildi: kap_eval_bimas.json")
print(f"   Context: {len(context):,} karakter")
print(f"   Soru sayısı: {len(qas)}")

✅ Q1: pos=31824, span='Kazablanka'
✅ Q2: pos=6615, span='Mısır ve Fas'
✅ Q3: pos=21489, span='721,1 Milyar TL net satış'
✅ Q4: pos=22081, span='14.473'
✅ Q5: pos=14720, span='Mahmud Muhammed Topbaş'
✅ Q6: pos=28610, span='%18,1'
✅ Q7: pos=108101, span='96 MW'
✅ Q8: pos=118854, span='101.663'
✅ Q10: pos=119522, span='1.434'
✅ Q11: pos=42470, span='Dost Süt'
✅ Q12: pos=106043, span='Harras (Gıda), \nActisoft (Temizlik), \nDaycare (Kişisel \nBakı'

Toplam: 11/12 soru başarılı

Hatalar:
  Q9: anchor bulunamadı → 'Eskişehir'de yer alan fabrika'

💾 Kaydedildi: kap_eval_bimas.json
   Context: 720,784 karakter
   Soru sayısı: 11


In [27]:
import re

# Apostrof olmadan anchor — daha güvenli
anchor = "Es Global Gıda Sanayi ve Ticaret A.Ş."
apos = context.find(anchor)

if apos == -1:
    # Whitespace-flex dene
    tokens = anchor.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, context)
    apos = m.start() if m else -1

print(f"Es Global anchor pos: {apos}")

# Anchor'dan sonra ilk "Eskişehir" geçişini bul
if apos != -1:
    spos = context.find("Eskişehir", apos)
    print(f"Eskişehir pos: {spos}")
    print(f"Çevre: {repr(context[spos:spos+80])}")

Es Global anchor pos: 51094
Eskişehir pos: 51328
Çevre: 'Eskişehir’de \nyer alan fabrikanın inşaatına 2022 yılında \nbaşlanmış olup 2024 yı'


In [28]:
q9 = {
    "id": "bimas_009",
    "question": "BİM'in Es Global fabrikası hangi şehirde bulunur?",
    "answers": {
        "text": ["Eskişehir"],
        "answer_start": [51328]
    }
}

# Q9'u 9. sıraya ekle (mevcut qas'ta Q1-Q8 + Q10-Q12 var)
qas_final = qas[:8] + [q9] + qas[8:]

# ID'leri yeniden düzenle (sıra korunsun diye)
for i, qa in enumerate(qas_final, 1):
    qa["id"] = f"bimas_{i:03d}"

bimas_record = {
    "title": "BIM_2025_Entegre_Faaliyet_Raporu",
    "context": context,
    "qas": qas_final
}

with open('/kaggle/working/kap_eval_bimas.json', 'w', encoding='utf-8') as f:
    json.dump(bimas_record, f, ensure_ascii=False, indent=2)

print(f"✅ Final: {len(qas_final)} soru")
print(f"💾 Kaydedildi: kap_eval_bimas.json\n")

# Sanity check — her ID ve soru görünsün
for qa in qas_final:
    print(f"  {qa['id']}: {qa['question'][:60]}... → {qa['answers']['text'][0][:40]}")

✅ Final: 12 soru
💾 Kaydedildi: kap_eval_bimas.json

  bimas_001: Fas'taki ilk BİM mağazası hangi şehirde açılmıştır?... → Kazablanka
  bimas_002: BİM hangi yurt dışı ülkelerde faaliyet göstermektedir?... → Mısır ve Fas
  bimas_003: BİM'in 31 Aralık 2025 itibarıyla net satışları kaç TL'dir?... → 721,1 Milyar TL net satış
  bimas_004: BİM'in 31 Aralık 2025 itibarıyla toplam mağaza sayısı kaçtır... → 14.473
  bimas_005: BİM'in Yönetim Kurulu Başkanı kimdir?... → Mahmud Muhammed Topbaş
  bimas_006: BİM'in hızlı tüketim ürünlerinde ciro pazar payı kaçtır?... → %18,1
  bimas_007: BİM'in 31 Aralık 2025 itibarıyla toplam GES kurulu gücü nedi... → 96 MW
  bimas_008: BİM'in 31 Aralık 2025 itibarıyla toplam çalışan sayısı kaçtı... → 101.663
  bimas_009: BİM'in Es Global fabrikası hangi şehirde bulunur?... → Eskişehir
  bimas_010: BİM'in 31 Aralık 2025 itibarıyla tedarikçi sayısı kaçtır?... → 1.434
  bimas_011: BİM'in Türkiye'de ilk kendi markalı ürünü nedir?... → Dost Süt
  bimas_012: FİLE'nin ke

In [29]:
# 1. Model'i ZIP'le, /kaggle/working/ içine koy
import shutil
shutil.make_archive('/kaggle/working/ft_model_backup', 'zip', 
                    '/kaggle/working/ft-bert-tr-tquad/')

# 2. Eval JSON'ları da yedekle
import json, glob, os
eval_files = glob.glob('/kaggle/working/kap_eval_*.json')
print(f"Bulunan eval dosyaları: {eval_files}")

# 3. Bunları Kaggle Dataset olarak save et (en güvenlisi)

Bulunan eval dosyaları: ['/kaggle/working/kap_eval_asels.json', '/kaggle/working/kap_eval_bimas.json']


In [30]:
import json

# Load
with open('/kaggle/working/kap_texts.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# CWENE full_text
text = data['CWENE']['full_text']
n = len(text)
print(f"Toplam karakter: {n}")
print(f"Yaklaşık kelime: {n // 5}\n")

# 4 çeyreğe böl
q_size = n // 4
quarters = [text[i*q_size:(i+1)*q_size] for i in range(4)]
# Son çeyrek kalan her şeyi alsın
quarters[3] = text[3*q_size:]

for i, q in enumerate(quarters, 1):
    print(f"{'='*50}")
    print(f"Q{i} ({len(q)} karakter) — ilk 300:")
    print(q[:300])
    print()

Toplam karakter: 95822
Yaklaşık kelime: 19164

Q1 (23955 karakter) — ilk 300:
 
1 
 
 
 
 
 
 


 
2 
 
İçindekiler 
1.ŞİRKET GENEL BİLGİLERİ ............................................................................................... 3 
2. SERMAYE, ORTAKLIK YAPISI, İMTİYAZLI PAYLAR VE KAR PAYI DAĞITIMI ....... 5 
3. ORGANİZASYON ŞEMASI ....................................

Q2 (23955 karakter) — ilk 300:
 EVA, POE, EPE Ham Madde Üretimi 
EVA, POE, EPE fotovoltaik güneş panellerinin üretiminde kullanılan ve cam, hücre ile 
backsheet’in 180°C sıcaklık altında birbirine yapışmasını sağlayan kimyasal bir üründür. Şirket, 
EVA üretimi için Antalya Organize Sanayi Bölgesi 3. Kısımdaki eski fabrika binasın

Q3 (23955 karakter) — ilk 300:
. Brüt kâr marjı ise %24,41’den %24,77’ye yükselerek 
sınırlı bir iyileşme göstermiştir. 
Faaliyet giderleri 1.212,2 milyon TL’den 1.573,7 milyon TL’ye yükselirken, FAVÖK 
(EBITDA) yaklaşık %15,3 artarak 2.751,7 milyon TL’den 3.173,7 milyon TL’ye ulaşmıştı

In [31]:
# Her çeyrekten 800'er karakter daha göster (300-1100 arası)
for i, q in enumerate(quarters, 1):
    print(f"{'='*50}")
    print(f"Q{i} — 300'den 1100'e:")
    print(q[300:1100])
    print()

Q1 — 300'den 1100'e:
............................................................. 8 
4. YÖNETİM KURULU, ÜST DÜZEY YÖNETİCİLERİ, PERSONEL BİLGİLERİ VE 
KOMİTELER ............................................................................................................................ 8 
5. GENEL KURULLAR VE ESAS SÖZLEŞMEDE YAPILAN DEĞİŞİKLİKLER ............. 11 
6. YÖNETİM KURULU ÜYELERİ VE ÜST DÜZEY YÖNETİCİLERE SAĞLANAN 
MALİ HAKLAR ....................................................................................................................... 11 
7. ŞİRKETİN ANA FAALİYETLERİ ..................................................................................... 12 
8. ARAŞTIRMA VE GELİŞTİRME ÇALIŞMALARI ........................................................... 20 
9. HUKUKİ VE SOSYAL KONULAR ......

Q2 — 300'den 1100'e:
ı yeniden 
inşa etmiş, gerekli makine ve ekipmanları temin ederek 2023 yılının ilk yarısında seri üretime 
Çeşitli Güç ve 
Teknolojilere Sahip
Fotovoltaik G

In [32]:
# Q1'in 1100-2500 arası — şirket genel bilgileri bölümü
print(quarters[0][1100:2500])

# Q3'ün 1100-2200 arası
print(quarters[2][1100:2200])


print("yeniiiiiii")
# Şirket genel bilgileri bölümü — Q1'in 2500-5000 arası
print(quarters[0][2500:5000])

.............................................................................. 21 
10. FİNANSAL DURUM HAKKINDA ÖZET BİLGİLER ................................................... 22 
11. FİNANSMAN KAYNAKLARININ GELİŞİMİ VE İŞLETMENİN UYGULADIĞI 
POLİTİKALAR ....................................................................................................................... 26 
12. GÜNEŞ ENERJİSİ SEKTÖRÜNE GENEL BAKIŞ ....................................................... 27 
13.TEŞVİKLER ....................................................................................................................... 39 
14.BAĞLI ORTAKLIKLAR .................................................................................................... 39 
15.İLİŞKİLİ TARAF İŞLEMLERİ HAKKINDA .................................................................. 42 
16.KURUMSAL YÖNETİM, SÜRDÜRÜLEBİLİRLİK VE GÖNÜLLÜLÜK ESASLI 
GİRİŞİMLER ............................................................................

In [33]:
import re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

spans = [
    "Tarzan Tarık Sarvan",
    "2010 yılında Antalya'da",
    "Orta Doğu Teknik Üniversitesi Güneş Enerjisi Araştırma ve Uygulama Merkezi (ODTÜ GÜNAM)",
    "fotovoltaik panel üreticisi ve EPC (Anahtar Teslim Proje) şirketi",
    "3.173.738.536",
    "2.215.087.500",
    "13,54%",
    "274.631 m²",
    "1.8 GW",
    "Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge'de",
    "UL 61730 ve CSA C22.2",
    "büyüme ve yatırım odaklı finansman kullanımının kaynak yapısındaki payının arttığını göstermektedir",
]

for s in spans:
    idx, found = find_flex(text, s)
    if idx == -1:
        print(f"❌ BULUNAMADI: {s[:50]}")
    else:
        # Unique mi?
        idx2, _ = find_flex(text, s, idx+1)
        unique = "✅ unique" if idx2 == -1 else f"⚠️ TEKRAR: {idx2}"
        print(f"{unique} | pos={idx} | '{found[:60]}'")

⚠️ TEKRAR: 7837 | pos=2402 | 'Tarzan Tarık Sarvan'
❌ BULUNAMADI: 2010 yılında Antalya'da
✅ unique | pos=4663 | 'Orta Doğu Teknik Üniversitesi Güneş Enerjisi Araştırma ve Uy'
✅ unique | pos=2646 | 'fotovoltaik panel üreticisi ve EPC (Anahtar Teslim Proje) şi'
⚠️ TEKRAR: 49303 | pos=47220 | '3.173.738.536'
⚠️ TEKRAR: 49449 | pos=47260 | '2.215.087.500'
✅ unique | pos=49494 | '13,54%'
✅ unique | pos=2844 | '274.631 m²'
✅ unique | pos=2936 | '1.8 
GW'
❌ BULUNAMADI: Antalya Organize Sanayi Bölgesi ve Antalya Serbest
⚠️ TEKRAR: 71927 | pos=21892 | 'UL 61730 ve 
CSA C22.2'
✅ unique | pos=49678 | 'büyüme ve yatırım 
odaklı finansman kullanımının kaynak yapı'


In [34]:
# Apostrof fix denemeleri
fixes = [
    "2010 yılında Antalya",           # 'da kesmek
    "Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge",  # 'de kesmek
]
for s in fixes:
    idx, found = find_flex(text, s)
    print(f"pos={idx} | '{found}'")

# Tekrar edenlerin context'i — hangi occurrence daha temiz?
for span, label in [("Tarzan Tarık Sarvan", "Tarzan"), 
                     ("3.173.738.536", "FAVÖK"),
                     ("2.215.087.500", "NetKar"),
                     ("UL 61730 ve", "UL")]:
    idx = -1
    while True:
        idx, found = find_flex(text, span, idx+1)
        if idx == -1:
            break
        print(f"{label} | pos={idx} | context: '{text[max(0,idx-60):idx+len(span)+60]}'")
    print()

pos=2515 | '2010 yılında Antalya'
pos=2758 | 'Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge'
Tarzan | pos=2402 | context: 'i ve Genel Bakış 
CW Enerji, kurucu ortak ve hâkim hissedar Tarzan Tarık Sarvan’ın Almanya’da edindiği bilgi 
ve deneyimleri Türkiye’de değ'
Tarzan | pos=7837 | context: 'ayı
Oy 
Hakkı
Grubu 
Tutar (TL)
Oran (%)
Oran (%)
Oran 
(%)
Tarzan Tarık Sarvan 
A 
215.004.493
19,94
49,74
70,05
B 
321.286.966
29,80
Volk'
Tarzan | pos=12488 | context: 'oyadı 
Görevi 
Görev Başlangıç 
Tarihi
Görev 
Bitiş 
Tarihi
Tarzan Tarık Sarvan 
Yönetim Kurulu Başkanı 
03.07.2025 03.07.2028
Volkan Yılma'
Tarzan | pos=13211 | context: 'oğrultusunda Yönetim Kurulu şu şekilde oluşturulmuştur: 
• 
Tarzan Tarık Sarvan – Yönetim Kurulu Başkanı 
• 
Volkan Yılmaz – Yönetim Kurulu'

FAVÖK | pos=47220 | context: 'ortisman Giderleri 
695.368.356
602.875.828
FAVÖK (EBITDA) 
3.173.738.536
2.751.714.128
Dönem Karı 
2.215.087.500
478.063.048


 
24 '
FAVÖK | pos=49303 | context: 'k Oranları 
Ka

In [35]:
# Tarzan pos=12488 — unique mi orada?
ctx_slice = text[12400:12650]
print(repr(ctx_slice))
print()

# 2010 unique mi?
idx, _ = find_flex(text, "2010 yılında Antalya")
idx2, _ = find_flex(text, "2010 yılında Antalya", idx+1)
print(f"2010: ilk={idx}, ikinci={idx2}")

# Antalya OSB unique mi?
idx, _ = find_flex(text, "Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge")
idx2, _ = find_flex(text, "Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge", idx+1)
print(f"AntalyaOSB: ilk={idx}, ikinci={idx2}")

' yetkileri haizdir. \n \nAdı Soyadı \nGörevi \nGörev Başlangıç \nTarihi\nGörev \nBitiş \nTarihi\nTarzan Tarık Sarvan \nYönetim Kurulu Başkanı \n03.07.2025 03.07.2028\nVolkan Yılmaz \nYönetim Kurulu Başkan Yardımcısı \n03.07.2025 03.07.2028\nMustafa Ayten \nYönetim K'

2010: ilk=2515, ikinci=-1
AntalyaOSB: ilk=2758, ikinci=-1


In [36]:
import json, re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

qa_raw = [
    ("CW Enerji hangi yıl ve şehirde kurulmuştur?",           "2010 yılında Antalya",          0),
    ("CW Enerji'nin Yönetim Kurulu Başkanı kimdir?",          "Tarzan Tarık Sarvan",            12400),
    ("CW Enerji hangi üniversiteyle iş birliği içindedir?",   "Orta Doğu Teknik Üniversitesi Güneş Enerjisi Araştırma ve Uygulama Merkezi (ODTÜ GÜNAM)", 0),
    ("CW Enerji hangi sektörde faaliyet göstermektedir?",     "fotovoltaik panel üreticisi ve EPC (Anahtar Teslim Proje) şirketi", 0),
    ("CW Enerji'nin 2025 yılı FAVÖK tutarı nedir?",           "3.173.738.536",                  49200),
    ("2025 yılında net dönem karı kaç TL'dir?",               "2.215.087.500",                  49400),
    ("2025 yılında net kâr marjı yüzde kaçtır?",              "13,54%",                         0),
    ("CW Enerji'nin toplam faaliyet alanı kaç m²'dir?",       "274.631 m²",                     0),
    ("CW Enerji'nin yıllık PV paneli üretim kapasitesi nedir?","1.8 GW",                        0),
    ("CW Enerji hangi lokasyonlarda faaliyet göstermektedir?","Antalya Organize Sanayi Bölgesi ve Antalya Serbest Bölge", 0),
    ("CW Enerji hangi Kuzey Amerika sertifikalarını almıştır?","UL 61730 ve CSA C22.2",         71900),
    ("Kaldıraç oranındaki artış neyi göstermektedir?",        "büyüme ve yatırım odaklı finansman kullanımının kaynak yapısındaki payının arttığını göstermektedir", 0),
]

qas = []
for q, span_query, start in qa_raw:
    idx, found = find_flex(text, span_query, start)
    if idx == -1:
        print(f"❌ {q}")
        continue
    qas.append({
        "id": f"CWENE_{len(qas)+1:02d}",
        "question": q,
        "answers": [{"text": found, "answer_start": idx}],
        "context_snippet": text[max(0,idx-100):idx+len(found)+100]
    })
    print(f"✅ {qas[-1]['id']} | '{found[:50]}'")

print(f"\nToplam: {len(qas)}/12")

with open('/kaggle/working/cwene_eval.json', 'w', encoding='utf-8') as f:
    json.dump({"company": "CWENE", "qas": qas}, f, ensure_ascii=False, indent=2)
print("Kaydedildi.")

✅ CWENE_01 | '2010 yılında Antalya'
✅ CWENE_02 | 'Tarzan Tarık Sarvan'
✅ CWENE_03 | 'Orta Doğu Teknik Üniversitesi Güneş Enerjisi Araşt'
✅ CWENE_04 | 'fotovoltaik panel üreticisi ve EPC (Anahtar Teslim'
✅ CWENE_05 | '3.173.738.536'
✅ CWENE_06 | '2.215.087.500'
✅ CWENE_07 | '13,54%'
✅ CWENE_08 | '274.631 m²'
✅ CWENE_09 | '1.8 
GW'
✅ CWENE_10 | 'Antalya Organize Sanayi Bölgesi ve Antalya Serbest'
✅ CWENE_11 | 'UL 61730 ve CSA C22.2'
✅ CWENE_12 | 'büyüme ve yatırım 
odaklı finansman kullanımının k'

Toplam: 12/12
Kaydedildi.


In [37]:
# Format'ı ASELS/BIMAS ile uyumlu hale getir
fixed_qas = []
for qa in qas:
    fixed_qas.append({
        "id": qa["id"].lower(),  # CWENE_01 → cwene_01 (tutarlılık)
        "question": qa["question"],
        "answers": {
            "text": [qa["answers"][0]["text"]],
            "answer_start": [qa["answers"][0]["answer_start"]]
        }
    })

cwene_record = {
    "title": "CWENE_2025_Faaliyet_Raporu",
    "context": text,  # ⚠️ Tüm full_text — bunu unutma
    "qas": fixed_qas
}

with open('/kaggle/working/kap_eval_cwene.json', 'w', encoding='utf-8') as f:
    json.dump(cwene_record, f, ensure_ascii=False, indent=2)

print(f"✅ Düzeltildi: {len(fixed_qas)} soru")
print(f"💾 Kaydedildi: kap_eval_cwene.json")
print(f"   Context length: {len(text):,} karakter")

# Sanity check — ilk soru
print(f"\nÖrnek (Q1):")
print(json.dumps(fixed_qas[0], ensure_ascii=False, indent=2))

# Offset doğrulama
q1 = fixed_qas[0]
spos = q1["answers"]["answer_start"][0]
expected = q1["answers"]["text"][0]
actual = text[spos:spos+len(expected)]
print(f"\nOffset check: {'✅' if actual == expected else '❌'}")
print(f"  Expected: {repr(expected[:50])}")
print(f"  Actual  : {repr(actual[:50])}")

✅ Düzeltildi: 12 soru
💾 Kaydedildi: kap_eval_cwene.json
   Context length: 95,822 karakter

Örnek (Q1):
{
  "id": "cwene_01",
  "question": "CW Enerji hangi yıl ve şehirde kurulmuştur?",
  "answers": {
    "text": [
      "2010 yılında Antalya"
    ],
    "answer_start": [
      2515
    ]
  }
}

Offset check: ✅
  Expected: '2010 yılında Antalya'
  Actual  : '2010 yılında Antalya'


In [39]:
text = data['KTLEV']['full_text']
n = len(text)
print(f"Toplam karakter: {n}")

q_size = n // 4
quarters = [text[i*q_size:(i+1)*q_size] for i in range(4)]
quarters[3] = text[3*q_size:]

for i, q in enumerate(quarters, 1):
    print(f"{'='*50}")
    print(f"Q{i} ({len(q)} karakter) — ilk 300:")
    print(q[:300])
    print()

Toplam karakter: 115162
Q1 (28790 karakter) — ilk 300:
 
 
 
FAALİYET 
RAPORU 
31 ARALIK 2025 
  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
             
 
 
             
 
 
 
 
 
 
 
 
 
 
 
 
 
 
                                             
                                               
 
 
 
 
 
 
 


 
 
İÇİNDEKİLER 
 
SERMAYE VE ORTAK

Q2 (28790 karakter) — ilk 300:
 ilkesi çerçevesinde pay sahiplerine aktarılmaktadır. Dönem 
içerisinde şirketimizle ilgili olan ve pay sahipliği haklarının kullanımını etkileyecek her türlü gelişme Sermaye Piyasası Kurulu’nun 
Kanun ve Tebliğleri çerçevesinde özel durum açıklaması ile Kamuyu Aydınlatma Platformunda (KAP) şirketim

Q3 (28790 karakter) — ilk 300:
rasyon ve tahsisat birim 
kayıtlarının uygunluğu kapsamındaki işlemler incelenerek riske maruz değerler tespit edilmektedir. Tüm şubelerin; sözleşme, 
teminat, muhasebe kayıtları, finansal tablo ve veriler, likidite hesapları mevcut politika ve prosedürler ile mevzuata uyumluluğ

In [40]:
# Q1: sermaye/ortaklık bölümü
print("=== Q1: 300-2000 ===")
print(quarters[0][300:2000])

# Q2: 0-800 (devam)
print("\n=== Q2: 0-800 ===")
print(quarters[1][0:800])

# Q3: 800-2000 (finansallar?)
print("\n=== Q3: 800-2000 ===")
print(quarters[2][800:2000])


print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")
print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")
print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")

# Q1 devamı — ortaklık yapısı tablosu
print("=== Q1: 2000-4000 ===")
print(quarters[0][2000:4000])

# Q3 finansallar
print("\n=== Q3: 2000-4000 ===")
print(quarters[2][2000:4000])

# Q4: Rakamlarla Katılımevim bölümü
print("\n=== Q4: 0-2000 ===")
print(quarters[3][0:2000])

print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")
print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")
print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")

print("=== Q1: 4000-6000 ===")
print(quarters[0][4000:6000])

print("\n=== Q2: 800-2500 ===")
print(quarters[1][800:2500])

=== Q1: 300-2000 ===
LIK YAPISI 
YÖNETİM KURULU YAPISI 
YK ÜYELERİNİN ORTAKLIK DIŞINDA YAPMIŞ OLDUĞU GÖREVLER 
ÜST DÜZEY YÖNETİCİLER 
İLİŞKİLİ TARAFLARA İLİŞKİN BİLGİLER 
BAĞLI ORTAKLIKLARA İLİŞKİN BİLGİLER 
YÖNETİM KURULU BÜNYESİNDE OLUŞTURULAN KOMİTELER 
PAY SAHİPLERİ 
KAMUOYU AYDINLATMA VE ŞEFFAFLIK 
MENFAAT SAHİPLERİ 
POLİTİKALAR 
YÖNETİM KURULUNUN FAALİYET ESASLARI 
İÇ DENETİM FAALİYETLERİ 
İNSAN KAYNAKLARI YÖNETİMİ 
RİSK YÖNETİMİ 
RAKAMLARLA KATILIMEVİM 
MALİ TABLOLAR 
ŞUBELERİMİZ 
 
 
 
 
 
 
 
 
 
 


 
 
1. 
KATILIMEVİM’İN HALKA ARZ SÜRECİ 
 
Katılımevim ilk halka arz edilen ve Borsa İstanbul Yıldız Pazar’da İlk işlem gören tasarruf finansman şirketi 
oldu. Katılımevim‘in halka arzı İnfo Yatırım aracılığıyla 6-7 Haziran 2023 Tarihlerinde “Borsada Satış” 
Yönetimiyle gerçekleşti. Gerçekleşen halka arzın büyüklüğü, Borsa İstanbul Birincil Piyasa ‘da pay başına 
13,43 TL sabit fiyatla 805,8 Milyon TL oldu. Halka arzda toplam 473 bin 340 yerli gerçek ve tüzel kişi yatırımcı, 
839 

In [41]:
print("=== Q3: 4000-6000 ===")
print(quarters[2][4000:6000])

print("\n=== Q4: 2000-4000 ===")
print(quarters[3][2000:4000])

print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")
print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")

print("=== Q4: 4000-6000 ===")
print(quarters[3][4000:6000])

=== Q3: 4000-6000 ===
e izlendiği bir süreç olan Tahsisat riski yönetimi, araç ve konut 
portföylerini kapsıyor. Tahsis aşamasında politika ve prosedürler çerçevesinde takip edilmektedir. 
• 
Operasyonel Risk 
Katılımevim 
deki 
tüm 
operasyonel 
riskler, 
risklerin 
tanımlanması, 
değerlendirilmesi, 
izlenmesi 
ve 
kontrol 
edilebilmesi/azaltılabilmesi unsurları çerçevesinde, Yönetim Kurulu ve Denetim Komitesinin gözetiminde yönetilmektedir. İç 
Denetim Biriminin ve İç Kontrol Biriminin operasyonel risklerin izlenmesi ve kontrol edilmesine yönelik her bir riske ait denetim 
durumu, etki ve olasılıkları bu matris içinde değerlendirilmektedir. 
Risk matrisi; İç Kontrol Birimi ve İç Denetim tarafından izleniyor, güncelleniyor ve gerçekleştirilen incelemelerde esas alınıyor. 
• 
İtibar Riski 
Katılımevim, yasal otoriteler, müşteriler ve diğer piyasa oyuncuları gözünde itibar riski oluşturabilecek her türlü işlem ve 
faaliyetten kaçınır, topluma, doğal çevreye ve insanlığa yararlı olmak iç

In [42]:
spans = [
    ("27 Eylül 2018 tarihinde İstanbul Ümraniye", 0),
    ("Serdar Turhan", 0),
    ("Ahmet ÖZCAN", 0),
    ("6-7 Haziran 2023", 0),
    ("13,43 TL", 0),
    ("805,8 Milyon TL", 0),
    ("474 bin 210 yatırımcı", 0),
    ("183.488.098.069", 0),
    ("142.110", 0),
    ("41.610", 0),
    ("135.443", 0),
    ("62 adet", 0),
]

text = data['KTLEV']['full_text']

for s, start in spans:
    idx, found = find_flex(text, s, start)
    if idx == -1:
        print(f"❌ BULUNAMADI: {s}")
        continue
    idx2, _ = find_flex(text, s, idx+1)
    status = "✅ unique" if idx2 == -1 else f"⚠️ TEKRAR pos={idx2}"
    print(f"{status} | pos={idx} | '{found[:60]}'")

✅ unique | pos=1465 | '27 Eylül 2018 tarihinde İstanbul Ümraniye'
⚠️ TEKRAR pos=5370 | pos=1762 | 'Serdar Turhan'
⚠️ TEKRAR pos=6292 | pos=4229 | 'Ahmet ÖZCAN'
✅ unique | pos=1006 | '6-7 Haziran 2023'
✅ unique | pos=1160 | '13,43 TL'
✅ unique | pos=1183 | '805,8 Milyon TL'
✅ unique | pos=1379 | '474 
bin 210 yatırımcı'
⚠️ TEKRAR pos=90233 | pos=89255 | '183.488.098.069'
⚠️ TEKRAR pos=90738 | pos=90389 | '142.110'
⚠️ TEKRAR pos=90770 | pos=90595 | '41.610'
⚠️ TEKRAR pos=90788 | pos=90484 | '135.443'
✅ unique | pos=5012 | '62 adet'


In [43]:
for span, label in [
    ("Serdar Turhan", "Turhan"),
    ("Ahmet ÖZCAN", "Ozcan"),
    ("183.488.098.069", "Sozlesme"),
    ("142.110", "SozlesmeAdet"),
    ("41.610", "Teslimat"),
    ("135.443", "AktifUye"),
]:
    idx = -1
    while True:
        idx, found = find_flex(text, span, idx+1)
        if idx == -1:
            break
        print(f"{label} | pos={idx} | '{text[max(0,idx-60):idx+len(span)+60]}'")
    print()

Turhan | pos=1762 | 'paydan oluşur. 
Şirkette gerçek kişi nihai hâkim pay sahibi Serdar Turhan’dır ve şirket hisselerinin (payların) tamamı nama 
yazılıdı'
Turhan | pos=5370 | 'arasında Anadolu Üniversitesi İktisat Bölümü’nde tamamlayan Serdar Turhan, iş hayatına 2010 
yılında inşaat sektöründe başlamıştır. T'
Turhan | pos=14729 | 'nım 
Pusula Finans Holding A.Ş. 
Ortak 
Ömer Burkay 
Ortak 
Serdar Turhan 
İlişkili Kişi 
Ahmet Özcan 
İlişkili Kişi 
Pusula Portföy '
Turhan | pos=19397 | 'lımevim Tasarruf Finansman A.Ş. 
    9.990.000.000  
99,9% 
Serdar Turhan 
             2.500.000  
0,0% 
Ahmet Özcan 
             2'

Ozcan | pos=4229 | 'im Kurulu Üyesi, Başkan Yardımcısı 
21.03.2025- 10.03.2028 
Ahmet ÖZCAN 
Yönetim Kurulu Üyesi, Genel Müdür 
21.03.2025- 10.03.2028 '
Ozcan | pos=6292 | 'yönetim kurulu başkan yardımcısı görevini yürütmektedir. 
 
Ahmet ÖZCAN- Yönetim Kurulu Üyesi, Genel Müdür 
Ahmet Özcan, lisans eği'
Ozcan | pos=12927 | 'tim Sağlık İnşaat 
San.Tic.ltd.Şti. / Ortak, Ş

In [44]:
# Bağımsız YK üyeleri bitişik mi?
idx, found = find_flex(text, "Hayri Gökhan ALPMAN")
print(repr(text[idx:idx+200]))

# Neden-sonuç adayı: bedelsiz sermaye artırımı gerekçesi
idx2, found2 = find_flex(text, "Bedelsiz Sermaye")
print(f"\nBedelsiz sermaye pos={idx2}")
print(repr(text[idx2:idx2+300]) if idx2 != -1 else "bulunamadı")

'Hayri Gökhan ALPMAN \nBağımsız Yönetim Kurulu Üyesi \n21.03.2025- 10.03.2028 \nOsman İlker SAVURAN \nBağımsız Yönetim Kurulu Üyesi \n21.03.2025- 10.03.2028 \n \nŞirket’in yönetim kurulu üyeleri 10.03.2025 ta'

Bedelsiz sermaye pos=3157
'Bedelsiz Sermaye Artırımı Sonrasında) \nOrtağın Adı-Soyadı Ticaret Unvanı \nSermayedeki Payı (TL) \n(Pay Adedi) \nSermaye Oranı  \n(%) \nOy Hakkı Oranı  \n(%) \nSERDAR TURHAN \n738.300.000 \n35,67 \n35,67 \nÖMER BURKAY \n193.200.000 \n9,33 \n9,33 \nDİĞER \n1.138.500.000 \n55 \n55 \nTOPLAM \n2.070.000.000 \n100 \n100 \n \n21'


In [45]:
# Bağlı ortaklıklar listesi var mıydı?
idx, _ = find_flex(text, "Bağlı Ortaklık")
print(repr(text[idx:idx+400]))

print("🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢")

# "tasarruf finansman" tanımı veya amacı
idx, _ = find_flex(text, "tasarruf finansman şirketi")
print(repr(text[max(0,idx-100):idx+300]))

'Bağlı Ortaklıkları \n \n31 Aralık 2025 Tarihinde Sona Eren Hesap Dönemine Ait \nKonsolide Finansal Bilgiler  \n\n\n \n \n  Katılımevim Tasarruf Finansman Anonim Şirketi ve Bağlı Ortaklıkları \n31 Aralık 2025 Tarihi İtibarıyla \nKonsolide Finansal Durum Tablosu (Bilanço) \nPara Birimi: Türk Lirası (“TL”) \n \n  \n \nBağımsız Denetimden Geçmiş \nBağımsız Denetimden Geçmiş \nAKTİF KALEMLER \n \nCari Dönem \nÖnceki Dönem'
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
'KA ARZ SÜRECİ \n \nKatılımevim ilk halka arz edilen ve Borsa İstanbul Yıldız Pazar’da İlk işlem gören tasarruf finansman şirketi \noldu. Katılımevim‘in halka arzı İnfo Yatırım aracılığıyla 6-7 Haziran 2023 Tarihlerinde “Borsada Satış” \nYönetimiyle gerçekleşti. Gerçekleşen halka arzın büyüklüğü, Borsa İstanbul Birincil Piyasa ‘da pay başına \n13,43 TL sabit fiyatla 805,8 Milyon TL oldu. Halka arzda top'


In [46]:
# Şubeler listesi var mıydı? (içindekiler'de "ŞUBELERİMİZ" vardı)
idx, _ = find_flex(text, "ŞUBELERİMİZ")
print("SUBELER:", repr(text[idx:idx+500]))
print()

# Komiteler listesi
idx, _ = find_flex(text, "Denetim Komitesi")
print("KOMİTE:", repr(text[idx:idx+400]))
print()

# Neden-sonuç: kaldıraç/risk ile ilgili bir gerekçe cümlesi
idx, _ = find_flex(text, "büyüme stratejilerini")
print("BUYUME:", repr(text[max(0,idx-50):idx+300]) if idx != -1 else "yok")
print()

# Politikalar bölümünde gerekçeli bir cümle?
idx, _ = find_flex(text, "pay sahiplerine aktarılmaktadır")
print("POLITIKA:", repr(text[max(0,idx-100):idx+200]))

SUBELER: 'ŞUBELERİMİZ \n \n \n \n \n \n \n \n \n \n \n\n\n \n \n1. \nKATILIMEVİM’İN HALKA ARZ SÜRECİ \n \nKatılımevim ilk halka arz edilen ve Borsa İstanbul Yıldız Pazar’da İlk işlem gören tasarruf finansman şirketi \noldu. Katılımevim‘in halka arzı İnfo Yatırım aracılığıyla 6-7 Haziran 2023 Tarihlerinde “Borsada Satış” \nYönetimiyle gerçekleşti. Gerçekleşen halka arzın büyüklüğü, Borsa İstanbul Birincil Piyasa ‘da pay başına \n13,43 TL sabit fiyatla 805,8 Milyon TL oldu. Halka arzda toplam 473 bin 340 yerli gerçek ve tüzel k'

KOMİTE: 'Denetim Komitesi, Riskin Erken Saptanması Komitesi ve Kurumsal Yönetim Komitesi, Yönetim Kurulu içinden görevlendirilen \nüyelerden oluşmaktadır. \n7.1 Riskin Erken Saptanması Komitesi \nBaşkanlığına Bağımsız Üye Osman İlker SAVURAN yürütmekte olup, komite üyesi olarak Hayri Gökhan   ALPMAN atanmışlardır. \n \nKomite \nKomite Üyeleri \nKomitedeki Görevi \nNiteliği \nRiskin Erken Saptanması \nKomitesi \nOsman'

BUYUME: 'rlama ve \nKurumsal İletişim Birim

In [47]:
for span in [
    "Denetim Komitesi, Riskin Erken Saptanması Komitesi ve Kurumsal Yönetim Komitesi",
    "eşitlik ilkesi",
]:
    idx, found = find_flex(text, span)
    idx2, _ = find_flex(text, span, idx+1)
    status = "✅ unique" if idx2 == -1 else f"⚠️ TEKRAR pos={idx2}"
    print(f"{status} | pos={idx} | '{found[:70]}'")

✅ unique | pos=24814 | 'Denetim Komitesi, Riskin Erken Saptanması Komitesi ve Kurumsal Yönetim'
✅ unique | pos=28783 | 'eşitlik ilkesi'


In [48]:
import json, re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

text = data['KTLEV']['full_text']

qa_raw = [
    ("Katılımevim hangi tarihte ve nerede kurulmuştur?",                          "27 Eylül 2018 tarihinde İstanbul Ümraniye", 0),
    ("Katılımevim'in gerçek kişi nihai hâkim pay sahibi kimdir?",                 "Serdar Turhan",                             1700),
    ("Katılımevim'in Genel Müdürü kimdir?",                                       "Ahmet ÖZCAN",                              4200),
    ("Katılımevim halka arzı hangi tarihte gerçekleşmiştir?",                     "6-7 Haziran 2023",                         0),
    ("Halka arzda pay başına sabit fiyat nedir?",                                 "13,43 TL",                                 0),
    ("Katılımevim halka arzının büyüklüğü nedir?",                                "805,8 Milyon TL",                          0),
    ("Halka arza katılan toplam yatırımcı sayısı kaçtır?",                        "474 bin 210 yatırımcı",                    0),
    ("2025 yılı toplam sözleşme hacmi kaç TL'dir?",                               "183.488.098.069",                          90200),
    ("2025 yılında kaç teslimat gerçekleşmiştir?",                                "41.610",                                   90550),
    ("2025 yılında kaç adet yönetim kurulu kararı alınmıştır?",                   "62 adet",                                  0),
    ("Yönetim Kurulu bünyesinde hangi komiteler oluşturulmuştur?",                "Denetim Komitesi, Riskin Erken Saptanması Komitesi ve Kurumsal Yönetim Komitesi", 0),
    ("Kamuya açıklanmış bilgiler hangi ilke çerçevesinde pay sahiplerine aktarılmaktadır?", "eşitlik ilkesi",                 0),
]

qas = []
for q, span_query, start in qa_raw:
    idx, found = find_flex(text, span_query, start)
    if idx == -1:
        print(f"❌ {q}")
        continue
    qas.append({
        "id": f"ktlev_{len(qas)+1:03d}",
        "question": q,
        "answers": {
            "text": [found],
            "answer_start": [idx]
        }
    })
    print(f"✅ ktlev_{len(qas):03d} | pos={idx} | '{found[:50]}'")

print(f"\nToplam: {len(qas)}/12")

for qa in qas:
    spos = qa["answers"]["answer_start"][0]
    expected = qa["answers"]["text"][0]
    actual = text[spos:spos+len(expected)]
    assert actual == expected, f"{qa['id']} offset mismatch!"
print("✅ Tüm offset'ler doğru")

output = {
    "title": "KTLEV_2025_Faaliyet_Raporu",
    "context": text,
    "qas": qas
}

with open('/kaggle/working/kap_eval_ktlev.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("💾 Kaydedildi: kap_eval_ktlev.json")

✅ ktlev_001 | pos=1465 | '27 Eylül 2018 tarihinde İstanbul Ümraniye'
✅ ktlev_002 | pos=1762 | 'Serdar Turhan'
✅ ktlev_003 | pos=4229 | 'Ahmet ÖZCAN'
✅ ktlev_004 | pos=1006 | '6-7 Haziran 2023'
✅ ktlev_005 | pos=1160 | '13,43 TL'
✅ ktlev_006 | pos=1183 | '805,8 Milyon TL'
✅ ktlev_007 | pos=1379 | '474 
bin 210 yatırımcı'
✅ ktlev_008 | pos=90233 | '183.488.098.069'
✅ ktlev_009 | pos=90595 | '41.610'
✅ ktlev_010 | pos=5012 | '62 adet'
✅ ktlev_011 | pos=24814 | 'Denetim Komitesi, Riskin Erken Saptanması Komitesi'
✅ ktlev_012 | pos=28783 | 'eşitlik ilkesi'

Toplam: 12/12
✅ Tüm offset'ler doğru
💾 Kaydedildi: kap_eval_ktlev.json


In [49]:
# Aday 2: Bedelli sermaye iptali
span2 = "mevcut güçlü finansal yapısı, artan kârlılığı, faaliyetlerinden sağlanan nakit akışı ve özkaynak seviyesi dikkate alınarak"
idx, found = find_flex(text, span2)
if idx == -1:
    print(f"❌ Aday2 bulunamadı")
else:
    idx2, _ = find_flex(text, span2, idx+1)
    print(f"{'✅ unique' if idx2==-1 else f'⚠️ TEKRAR pos={idx2}'} | pos={idx}")
    print(f"found: '{found}'")
print()

# Aday 1: Birden fazla komite
span1 = "mevzuat gereği her komitede bulunma zorunluluğu sebebiyle bir yönetim kurulu üyesinin birden fazla komitede görev alması durumu ortaya çıkmıştır"
idx, found = find_flex(text, span1)
if idx == -1:
    print(f"❌ Aday1 bulunamadı")
else:
    idx2, _ = find_flex(text, span1, idx+1)
    print(f"{'✅ unique' if idx2==-1 else f'⚠️ TEKRAR pos={idx2}'} | pos={idx}")
    print(f"found: '{found}'")

✅ unique | pos=71371
found: 'mevcut güçlü finansal yapısı, artan kârlılığı, faaliyetlerinden sağlanan nakit akışı ve özkaynak 
seviyesi dikkate alınarak'

✅ unique | pos=26812
found: 'mevzuat gereği her komitede bulunma zorunluluğu sebebiyle bir yönetim kurulu üyesinin birden 
fazla komitede görev alması durumu ortaya çıkmıştır'


In [50]:
import json, re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

text = data['KTLEV']['full_text']

qa_raw = [
    ("Katılımevim hangi tarihte ve nerede kurulmuştur?",                          "27 Eylül 2018 tarihinde İstanbul Ümraniye", 0),
    ("Katılımevim'in gerçek kişi nihai hâkim pay sahibi kimdir?",                 "Serdar Turhan",                             1700),
    ("Katılımevim'in Genel Müdürü kimdir?",                                       "Ahmet ÖZCAN",                              4200),
    ("Katılımevim halka arzı hangi tarihte gerçekleşmiştir?",                     "6-7 Haziran 2023",                         0),
    ("Halka arzda pay başına sabit fiyat nedir?",                                 "13,43 TL",                                 0),
    ("Katılımevim halka arzının büyüklüğü nedir?",                                "805,8 Milyon TL",                          0),
    ("Halka arza katılan toplam yatırımcı sayısı kaçtır?",                        "474 bin 210 yatırımcı",                    0),
    ("2025 yılı toplam sözleşme hacmi kaç TL'dir?",                               "183.488.098.069",                          90200),
    ("2025 yılında kaç teslimat gerçekleşmiştir?",                                "41.610",                                   90550),
    ("2025 yılında kaç adet yönetim kurulu kararı alınmıştır?",                   "62 adet",                                  0),
    ("Yönetim Kurulu bünyesinde hangi komiteler oluşturulmuştur?",                "Denetim Komitesi, Riskin Erken Saptanması Komitesi ve Kurumsal Yönetim Komitesi", 0),
    ("Bedelli sermaye artırımı süreci neden iptal edilmiştir?",                   "mevcut güçlü finansal yapısı, artan kârlılığı, faaliyetlerinden sağlanan nakit akışı ve özkaynak seviyesi dikkate alınarak", 71300),
]

qas = []
for q, span_query, start in qa_raw:
    idx, found = find_flex(text, span_query, start)
    if idx == -1:
        print(f"❌ {q}")
        continue
    qas.append({
        "id": f"ktlev_{len(qas)+1:03d}",
        "question": q,
        "answers": {
            "text": [found],
            "answer_start": [idx]
        }
    })
    print(f"✅ ktlev_{len(qas):03d} | pos={idx} | '{found[:50]}'")

print(f"\nToplam: {len(qas)}/12")

for qa in qas:
    spos = qa["answers"]["answer_start"][0]
    expected = qa["answers"]["text"][0]
    actual = text[spos:spos+len(expected)]
    assert actual == expected, f"{qa['id']} offset mismatch!"
print("✅ Tüm offset'ler doğru")

output = {
    "title": "KTLEV_2025_Faaliyet_Raporu",
    "context": text,
    "qas": qas
}

with open('/kaggle/working/kap_eval_ktlev.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("💾 Kaydedildi: kap_eval_ktlev.json")

✅ ktlev_001 | pos=1465 | '27 Eylül 2018 tarihinde İstanbul Ümraniye'
✅ ktlev_002 | pos=1762 | 'Serdar Turhan'
✅ ktlev_003 | pos=4229 | 'Ahmet ÖZCAN'
✅ ktlev_004 | pos=1006 | '6-7 Haziran 2023'
✅ ktlev_005 | pos=1160 | '13,43 TL'
✅ ktlev_006 | pos=1183 | '805,8 Milyon TL'
✅ ktlev_007 | pos=1379 | '474 
bin 210 yatırımcı'
✅ ktlev_008 | pos=90233 | '183.488.098.069'
✅ ktlev_009 | pos=90595 | '41.610'
✅ ktlev_010 | pos=5012 | '62 adet'
✅ ktlev_011 | pos=24814 | 'Denetim Komitesi, Riskin Erken Saptanması Komitesi'
✅ ktlev_012 | pos=71371 | 'mevcut güçlü finansal yapısı, artan kârlılığı, faa'

Toplam: 12/12
✅ Tüm offset'ler doğru
💾 Kaydedildi: kap_eval_ktlev.json


In [51]:
import json

with open('/kaggle/working/kap_texts.json', 'r', encoding='utf-8') as f:
    kap = json.load(f)

ticker = 'YEOTK'
full_text = kap[ticker]['full_text']
n = len(full_text)
print(f"YEOTK full_text uzunluğu: {n} karakter")

q1 = full_text[:n//4]
q2 = full_text[n//4:n//2]
q3 = full_text[n//2:3*n//4]
q4 = full_text[3*n//4:]

for i, (label, chunk) in enumerate(zip(['Q1','Q2','Q3','Q4'], [q1,q2,q3,q4]), 1):
    start_idx = (i-1) * (n//4)
    print(f"\n--- {label} (offset {start_idx}) ---")
    print(repr(chunk[:300]))

YEOTK full_text uzunluğu: 159634 karakter

--- Q1 (offset 0) ---
'YEO 2025 Faaliyet Raporu\n1\n #BizceMümkün\n\n\nYEO 2025 Faaliyet Raporu\nYEO 2025 Faaliyet Raporu\n2\n3\n #BizceMümkün\n #BizceMümkün\n%100 Yeo Teknoloji iştiraki \nolan Reap Battery, İstanbul \nTuzla’daki ileri teknoloji robotik \nhatlara sahip tesislerinde LFP \n(Lityum-Demir-Fosfat) bazlı \nyerli enerji depolam'

--- Q2 (offset 39908) ---
'ra Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü’nü bitiren Mehmet Öğütçü, \nLondon School of Economics (LSE)’den Uluslararası Ekonomi üzerine yüksek lisans derecesi aldı. \nBruges’deki College d’Europe’da Avrupa Yönetimi alanında yüksek lisans yapan Öğütçü, halen LSE, \nReading'

--- Q3 (offset 79816) ---
' \nmaliyet etkin konteyner tipi enerji depolama sistemleri\nFarklı uygulama alanları sınırsız çözümler sunmak için \n373 – 418 kWh kapasiteli modüler kabin tipi \nenerji depolama sistemleri\nTelekom, endüstriyel taşıma araçları, hızlı araç şarj \nistasyon

In [52]:
print(repr(full_text[119724:121000]))

' \n 1.231.719.743 \n- İlişkili Taraflara Ticari Borçlar\nNot.10,38\n 124.039.864 \n 574.348.242 \nÇalışanlara Sağlanan Faydalar Kapsamında Borçlar\nNot.21\n 266.814.052 \n 112.875.191 \nDiğer Borçlar\nNot.11\n  1.195.085.232\n 383.602.176 \n- İlişkili Olmayan Taraflara Diğer Borçlar\nNot.11,38\n  67.733.890 \n 26.443.581 \n- İlişkili Taraflara Diğer Borçlar\nNot.11,38\n 1.127.351.342 \n 357.158.595 \nMüşteri Sözleşmelerinden Doğan Yükümlülükler\nNot.12\n  404.662.916 \n - \nErtelenmiş Gelirler\nNot.15\n 4.275.646.744 \n 1.244.013.542 \nDönem Karı Vergi Yükümlülüğü\nNot.26\n 38.878.332 \n 29.030.216 \nKısa Vadeli Karşılıklar\nNot.23\n 47.042.798 \n 36.455.187 \n- Diğer Kısa Vadeli Karşılıklar\n 12.135.700 \n 13.080.663 \n- Çalışanlara Sağlanan Faydalara İlişkin Kısa Vadeli Karşılıklar\n 34.907.098 \n 23.374.524 \nToplam\n 14.706.304.456 \n  5.397.995.284  \nSatış Amacıyla Elde Tutulan Duran Varlıklara İlişkin Yük.\nNot.34\n  3.205.883.973 \n - \nUzun Vadeli Yükümlülükler\n  4.237.626.0

In [53]:
# Hedef bölgeleri geniş tut
sections = {
    "Q1_reap": (0, 2000),
    "Q2_ogutcu": (39000, 43000),
    "Q3_products": (79000, 83000),
    "Q4_financial": (119000, 122000),
}

for label, (s, e) in sections.items():
    print(f"\n{'='*20} {label} ({s}-{e}) {'='*20}")
    print(repr(full_text[s:e]))


==================== Q1_reap (0-2000) ====================
"YEO 2025 Faaliyet Raporu\n1\n #BizceMümkün\n\n\nYEO 2025 Faaliyet Raporu\nYEO 2025 Faaliyet Raporu\n2\n3\n #BizceMümkün\n #BizceMümkün\n%100 Yeo Teknoloji iştiraki \nolan Reap Battery, İstanbul \nTuzla’daki ileri teknoloji robotik \nhatlara sahip tesislerinde LFP \n(Lityum-Demir-Fosfat) bazlı \nyerli enerji depolama sistemleri \n(EDS) üretimine  başladı. Yıllık \n5 GWh üretim kapasitesine \nsahip tesislerde Türkiye’nin yanı \nsıra ABD, Avrupa, Orta Doğu, \nOrta Asya ve Afrika pazarlarına \nyönelik ihracat hedefleniyor.\n2025\n‘‘\n‘‘\n\n\nYEO 2025 Faaliyet Raporu\nYEO 2025 Faaliyet Raporu\n4\n5\n #BizceMümkün\n #BizceMümkün\nYEO, İKİ YIL ÜST ÜSTE \nFORTUNE 500 LİSTESİNDE!\nDünyanın en prestijli iş dünyası listelerinden biri olan Fortune 500'deki yerimizi iki yıl üst üste \nkorumanın haklı gururunu yaşıyoruz. Sürdürülebilir büyüme yolculuğumuzdaki bu istikrar, \ngelecekteki hedeﬂerimiz için en büyük motivasyon kaynağımız.\nBu i

In [54]:
import re

keywords = ["neden", "sebebiyle", "kaynaklanmaktadır", "nedeniyle", "amacıyla"]

for kw in keywords:
    for m in re.finditer(kw, full_text, re.IGNORECASE):
        start = max(0, m.start()-150)
        end = min(len(full_text), m.end()+150)
        print(f"\n['{kw}' @ {m.start()}]")
        print(repr(full_text[start:end]))
        print("---")


['neden' @ 13231]
'arı ve \nmesken tipi depolama projeleri ile portföyünü genişletmektedir. YEO Teknoloji’nin \ngüneş enerjisi ve dağıtık enerji üretimi alanında kapasite nedeniyle daha geniş bir \nkitleye hizmet vermek için iştirak ettiği kuruluştur.\n\n\nYEO 2025 Faaliyet Raporu\nYEO 2025 Faaliyet Raporu\n18\n19\n #BizceMümkün\n #B'
---

['neden' @ 91634]
'zda ise 2 projemiz \nteknik ve bilimsel değerlendirme süreçlerinde eşik üstü puan alarak \nbaşarılı bulunmuş; ancak ilgili çağrılardaki bütçe kısıtları nedeniyle \nfonlanamamıştır. Bu durum, projelerimizin kalite ve rekabet gücünü ortaya \nkoymakta olup, uluslararası standartlarda yetkin Ar-Ge kapasitemizi t'
---

['neden' @ 115396]
'ıklar gibi borç baskısı yaratmayan\nyükümlülüklerden oluşmaktadır.\nFaaliyetlerden elde edilen gelirlerin işletme sermayesi \nfinansmanı da kullanılması nedeniyle pozitif seyir \ndevam etmektedir.\nGES projeler, yurt dışı projeler ve enerji depolama \nbüyümede etkili olacak.\nYurt dışındaki projelerde

In [55]:
import re

def find_flex(ctx, clean_span, start=0):
    tokens = clean_span.split()
    pattern = r'\s+'.join(re.escape(t) for t in tokens)
    m = re.search(pattern, ctx[start:])
    if m:
        return start + m.start(), m.group(0)
    return -1, None

context = kap['YEOTK']['full_text']

candidates = [
    ("yeotk_001", "Tuzla",                                                  "İstanbul"),
    ("yeotk_002", "Lityum-Demir-Fosfat",                                    "LFP"),
    ("yeotk_003", "5 GWh",                                                  "Yıllık"),
    ("yeotk_004", "iki",                                                    "Fortune 500'deki yerimizi iki yıl üst üste"),
    ("yeotk_005", "Ankara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü", "bitiren Mehmet"),
    ("yeotk_006", "bütçe kısıtları",                                        "çağrılardaki"),
    ("yeotk_007", "DOMiNO Ventures",                                        "girişim sermayesi fonu"),
    ("yeotk_008", "373 – 418 kWh",                                          "modüler kabin tipi"),
    ("yeotk_009", "3.73 – 5 MWh",                                           "konteyner tipi enerji depolama"),
    ("yeotk_010", "Telekom, endüstriyel taşıma araçları, hızlı araç şarj \nistasyonları", "için"),
    ("yeotk_011", "4.275.646.744",                                          "Ertelenmiş Gelirler"),
    ("yeotk_012", "3.248.494.882",                                          "Uzun Vadeli Borçlanmalar"),
]

results = []
for qid, span, anchor in candidates:
    anchor_pos = context.find(anchor)
    if anchor_pos == -1:
        print(f"❌ {qid}: anchor bulunamadı: {repr(anchor)}")
        continue
    pos, found = find_flex(context, span, anchor_pos)
    if pos == -1:
        print(f"❌ {qid}: span bulunamadı: {repr(span)}")
        continue
    # verify
    actual = context[pos:pos+len(found)]
    print(f"✅ {qid} @ {pos}: {repr(found)}")
    results.append((qid, span, found, pos))

✅ yeotk_001 @ 183: 'Tuzla'
✅ yeotk_002 @ 252: 'Lityum-Demir-Fosfat'
✅ yeotk_003 @ 348: '5 GWh'
✅ yeotk_004 @ 728: 'iki'
❌ yeotk_005: span bulunamadı: 'Ankara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü'
✅ yeotk_006 @ 91618: 'bütçe kısıtları'
✅ yeotk_007 @ 41148: 'DOMiNO Ventures'
❌ yeotk_008: span bulunamadı: '373 – 418 kWh'
❌ yeotk_009: span bulunamadı: '3.73 – 5 MWh'
✅ yeotk_010 @ 80003: 'Telekom, endüstriyel taşıma araçları, hızlı araç şarj \nistasyonları'
✅ yeotk_011 @ 120205: '4.275.646.744'
✅ yeotk_012 @ 120723: '3.248.494.882'


In [56]:
# 005: anchor'u bul, çevresini gör
anchor_005 = context.find("bitiren Mehmet")
print(f"anchor_005 @ {anchor_005}")
print(repr(context[anchor_005-200:anchor_005+100]))

# 008/009: gerçek dash karakterini tespit et
idx_008 = context.find("373")
print(f"\n008 bölgesi @ {idx_008}:")
print(repr(context[idx_008:idx_008+30]))
print("hex:", context[idx_008:idx_008+20].encode('utf-8').hex())

idx_009 = context.find("3.73")
print(f"\n009 bölgesi @ {idx_009}:")
print(repr(context[idx_009:idx_009+20]))
print("hex:", context[idx_009:idx_009+20].encode('utf-8').hex())

anchor_005 @ 39984
'indsor Energy \nGroup, European Policy Forum, The Oil Council ve Beijing Energy Club\n/ Uluslararası Danışma Kurulu Üyesi\nAnkara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü’nü bitiren Mehmet Öğütçü, \nLondon School of Economics (LSE)’den Uluslararası Ekonomi üzerine yüksek lis'

008 bölgesi @ 79931:
'373 – 418 kWh kapasiteli modül'
hex: 33373320e2809320343138206b5768206b6170617369

009 bölgesi @ 79775:
'3.73 – 5 MWh kapasit'
hex: 332e373320e280932035204d5768206b617061736974


In [57]:
# 005: anchor'u span'den önce gelen metne çek
anchor_005b = "Uluslararası Danışma Kurulu Üyesi\nAnkara"
pos_a = context.find(anchor_005b)
pos5, found5 = find_flex(context, "Ankara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü", pos_a)
print(f"005: {pos5} → {repr(found5)}")

# 008/009: en-dash ile dene (U+2013)
span_008 = "373 \u2013 418 kWh"
span_009 = "3.73 \u2013 5 MWh"

pos8, found8 = find_flex(context, span_008, context.find("modüler kabin tipi"))
pos9, found9 = find_flex(context, span_009, context.find("konteyner tipi enerji depolama"))
print(f"008: {pos8} → {repr(found8)}")
print(f"009: {pos9} → {repr(found9)}")

005: 39904 → 'Ankara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü'
008: -1 → None
009: -1 → None


In [58]:
# Direkt search, anchor yok
import re

for label, span in [("008", "373 \u2013 418 kWh"), ("009", "3.73 \u2013 5 MWh")]:
    m = re.search(re.escape(span), context)
    if m:
        print(f"✅ {label} @ {m.start()}: {repr(m.group())}")
    else:
        # PDF'deki tam karakteri bul
        idx = context.find("373")
        print(f"❌ {label} — ham metin: {repr(context[idx:idx+15])}")
        # karakter karakter hex
        for i, ch in enumerate(context[idx:idx+10]):
            print(f"  [{i}] {repr(ch)} = U+{ord(ch):04X}")

✅ 008 @ 79931: '373 – 418 kWh'
✅ 009 @ 79775: '3.73 – 5 MWh'


In [59]:
import json

context = kap['YEOTK']['full_text']

qas = [
    {"id": "yeotk_001", "question": "Reap Battery tesisi İstanbul'un hangi ilçesinde yer almaktadır?",
     "answers": {"text": ["Tuzla"], "answer_start": [183]}},
    {"id": "yeotk_002", "question": "LFP kısaltması neyin kısaltmasıdır?",
     "answers": {"text": ["Lityum-Demir-Fosfat"], "answer_start": [252]}},
    {"id": "yeotk_003", "question": "Reap Battery tesislerinin yıllık üretim kapasitesi ne kadardır?",
     "answers": {"text": ["5 GWh"], "answer_start": [348]}},
    {"id": "yeotk_004", "question": "YEO, Fortune 500 listesinde kaç yıl üst üste yer almıştır?",
     "answers": {"text": ["iki"], "answer_start": [728]}},
    {"id": "yeotk_005", "question": "Mehmet Öğütçü hangi üniversitede Uluslararası İlişkiler eğitimi aldı?",
     "answers": {"text": ["Ankara Üniversitesi Siyasal Bilgiler Fakültesi Uluslararası İlişkiler Bölümü"], "answer_start": [39904]}},
    {"id": "yeotk_006", "question": "Başarılı bulunan AR-GE projeleri neden fonlanamamıştır?",
     "answers": {"text": ["bütçe kısıtları"], "answer_start": [91618]}},
    {"id": "yeotk_007", "question": "Mustafa Kopuk hangi girişim sermayesi fonunun kurucusu ve yönetici ortağıdır?",
     "answers": {"text": ["DOMiNO Ventures"], "answer_start": [41148]}},
    {"id": "yeotk_008", "question": "Modüler kabin tipi enerji depolama sistemlerinin kapasite aralığı nedir?",
     "answers": {"text": ["373 \u2013 418 kWh"], "answer_start": [79931]}},
    {"id": "yeotk_009", "question": "Şebeke ölçekli konteyner tipi enerji depolama sistemlerinin kapasite aralığı nedir?",
     "answers": {"text": ["3.73 \u2013 5 MWh"], "answer_start": [79775]}},
    {"id": "yeotk_010", "question": "Özel tasarım enerji depolama sistemleri hangi uygulamalar için üretilmektedir?",
     "answers": {"text": ["Telekom, endüstriyel taşıma araçları, hızlı araç şarj \nistasyonları"], "answer_start": [80003]}},
    {"id": "yeotk_011", "question": "31.12.2025 itibarıyla ertelenmiş gelirler ne kadardır?",
     "answers": {"text": ["4.275.646.744"], "answer_start": [120205]}},
    {"id": "yeotk_012", "question": "31.12.2025 itibarıyla uzun vadeli borçlanmalar ne kadardır?",
     "answers": {"text": ["3.248.494.882"], "answer_start": [120723]}},
]

# Sanity check
for qa in qas:
    spos = qa["answers"]["answer_start"][0]
    expected = qa["answers"]["text"][0]
    actual = context[spos:spos+len(expected)]
    assert actual == expected, f"❌ {qa['id']} offset mismatch!\n  expected: {repr(expected)}\n  actual:   {repr(actual)}"
print("✅ Tüm offset'ler doğru")

data = {
    "title": "YEOTK_2025_Faaliyet_Raporu",
    "context": context,
    "qas": qas
}

with open('/kaggle/working/kap_eval_yeotk.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ kap_eval_yeotk.json kaydedildi")

✅ Tüm offset'ler doğru
✅ kap_eval_yeotk.json kaydedildi


In [60]:
import os
files = sorted(os.listdir('/kaggle/working/'))
print(f"/kaggle/working/ içeriği ({len(files)} öğe):\n")
for f in files:
    full = f'/kaggle/working/{f}'
    if os.path.isdir(full):
        sub = os.listdir(full)
        print(f"  📁 {f}/  ({len(sub)} öğe)")
    else:
        size = os.path.getsize(full) / 1024
        print(f"  📄 {f}  ({size:.1f} KB)")

/kaggle/working/ içeriği (10 öğe):

  📁 .virtual_documents/  (1 öğe)
  📄 cwene_eval.json  (6.3 KB)
  📁 ft-bert-tr-tquad/  (6 öğe)
  📄 ft_model_backup.zip  (1512670.2 KB)
  📄 kap_eval_asels.json  (254.1 KB)
  📄 kap_eval_bimas.json  (799.6 KB)
  📄 kap_eval_cwene.json  (107.2 KB)
  📄 kap_eval_ktlev.json  (129.2 KB)
  📄 kap_eval_yeotk.json  (178.1 KB)
  📄 kap_texts.json  (2907.8 KB)


In [61]:
import json, glob, os
eval_files = glob.glob('/kaggle/working/kap_eval_*.json')
print(f"Bulunan eval dosyaları: {eval_files}")

Bulunan eval dosyaları: ['/kaggle/working/kap_eval_asels.json', '/kaggle/working/kap_eval_yeotk.json', '/kaggle/working/kap_eval_ktlev.json', '/kaggle/working/kap_eval_bimas.json', '/kaggle/working/kap_eval_cwene.json']


In [62]:
from pathlib import Path
OUTPUT_DIR = "/kaggle/working/ft-bert-tr-tquad"
MODEL_EXISTS = (Path(OUTPUT_DIR) / "model.safetensors").exists()

if MODEL_EXISTS:
    print("✓ Model zaten var, eğitim atlanıyor")
else:
    # eğitim kodu (15 dk koşar)
    ...

✓ Model zaten var, eğitim atlanıyor


**MERGE'E GEÇİYORUZ**,
**ADIM-1**

In [63]:
import json
from datasets import Dataset

tickers = ['asels', 'bimas', 'cwene', 'ktlev', 'yeotk']
flat_rows = []
per_company_stats = {}

for t in tickers:
    with open(f'/kaggle/working/kap_eval_{t}.json', 'r', encoding='utf-8') as f:
        record = json.load(f)
    
    for qa in record['qas']:
        flat_rows.append({
            'id': qa['id'],
            'title': record['title'],
            'context': record['context'],
            'question': qa['question'],
            'answers': {
                'text': qa['answers']['text'],
                'answer_start': qa['answers']['answer_start']
            }
        })
    
    per_company_stats[t] = len(record['qas'])

# Sanity
print(f"Toplam soru: {len(flat_rows)}\n")
for t, n in per_company_stats.items():
    print(f"  {t.upper()}: {n}")

# Offset doğrulama
errors = 0
for row in flat_rows:
    spos = row['answers']['answer_start'][0]
    expected = row['answers']['text'][0]
    actual = row['context'][spos:spos+len(expected)]
    if actual != expected:
        print(f"❌ {row['id']}")
        errors += 1

print(f"\n{'✅' if errors == 0 else '❌'} Offset check: {len(flat_rows)-errors}/{len(flat_rows)}")

# HF Dataset olarak kaydet
kap_eval = Dataset.from_list(flat_rows)
kap_eval.save_to_disk('/kaggle/working/kap_eval_dataset')

print(f"\n💾 HF Dataset: /kaggle/working/kap_eval_dataset")
print(f"   Features: {kap_eval.features}")
print(f"   Size: {len(kap_eval)} rows")

Toplam soru: 60

  ASELS: 12
  BIMAS: 12
  CWENE: 12
  KTLEV: 12
  YEOTK: 12

✅ Offset check: 60/60


Saving the dataset (0/1 shards):   0%|          | 0/60 [00:00<?, ? examples/s]


💾 HF Dataset: /kaggle/working/kap_eval_dataset
   Features: {'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'answer_start': List(Value('int64')), 'text': List(Value('string'))}}
   Size: 60 rows


In [64]:
# === Cell KAP-EVAL: Run ft_model on KAP eval set ===
from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch
import numpy as np

# 1. KAP eval dataset'i yükle
kap_eval = load_from_disk('/kaggle/working/kap_eval_dataset')
print(f"KAP eval: {len(kap_eval)} soru")

# 2. ft_tokenizer ile tokenize (prepare_validation_features Cell 5'te tanımlı)
print("\nTokenizing KAP eval (sliding window)...")
kap_features = kap_eval.map(
    prepare_validation_features,
    batched=True,
    remove_columns=kap_eval.column_names,
    desc="Tokenizing KAP",
)
print(f"KAP features: {len(kap_features)} (expansion: {len(kap_features)/len(kap_eval):.1f}x)")

# 3. Şirket bazında feature dağılımı (sliding window expansion'ı gör)
from collections import Counter
example_to_ticker = {ex['id']: ex['id'].split('_')[0] for ex in kap_eval}
feat_per_ticker = Counter(example_to_ticker[feat['example_id']] for feat in kap_features)
print(f"\nFeature dağılımı:")
for ticker, count in sorted(feat_per_ticker.items()):
    n_q = sum(1 for ex in kap_eval if ex['id'].startswith(ticker))
    print(f"  {ticker.upper()}: {count} features / {n_q} soru = {count/n_q:.0f}x")

# 4. ft_model inference (run_inference Cell 6b'de tanımlı)
# run_inference yoksa yeniden tanımlayalım — Cell 6b'den kopya
def run_inference(model_, features_, batch_size=32):
    model_.eval()
    model_inputs = features_.remove_columns(["offset_mapping", "example_id"])
    model_inputs.set_format("torch")
    loader = DataLoader(model_inputs, batch_size=batch_size, shuffle=False)
    all_start, all_end = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model_(**batch)
            all_start.append(out.start_logits.cpu().numpy())
            all_end.append(out.end_logits.cpu().numpy())
    return np.concatenate(all_start, axis=0), np.concatenate(all_end, axis=0)

print("\nRunning ft_model inference on KAP features...")
kap_start_logits, kap_end_logits = run_inference(ft_model, kap_features, batch_size=32)
print(f"Logits: start {kap_start_logits.shape}, end {kap_end_logits.shape}")

# 5. Postprocess → text predictions
# KAP examples'ı dict listesine çevir (postprocess fonksiyonu bunu bekliyor)
kap_examples = [
    {
        "id": ex["id"],
        "question": ex["question"],
        "context": ex["context"],
        "answers": [
            {"text": t, "answer_start": s}
            for t, s in zip(ex["answers"]["text"], ex["answers"]["answer_start"])
        ],
    }
    for ex in kap_eval
]

print("\nPostprocessing logits → text predictions...")
kap_predictions = postprocess_qa_predictions(
    examples=kap_examples,
    features=kap_features,
    all_start_logits=kap_start_logits,
    all_end_logits=kap_end_logits,
    n_best_size=20,
    max_answer_length=80,  # KAP'ta uzun list span'ler var (ASELS bağlı ortaklıklar 252 char)
)

# 6. SQuAD metric (squad_metric Cell 4'te yüklendi)
preds_formatted = [{"id": str(eid), "prediction_text": ptext}
                   for eid, ptext in kap_predictions.items()]
refs_formatted = [{"id": str(ex["id"]),
                   "answers": {
                       "text": [a["text"] for a in ex["answers"]],
                       "answer_start": [int(a["answer_start"]) for a in ex["answers"]],
                   }}
                  for ex in kap_examples]

overall_results = squad_metric.compute(predictions=preds_formatted, references=refs_formatted)

print("\n" + "="*60)
print("KAP EVAL — OVERALL RESULTS (ft_model on KAP custom eval)")
print("="*60)
print(f"Exact Match : {overall_results['exact_match']:.2f}")
print(f"F1          : {overall_results['f1']:.2f}")
print(f"\nKarşılaştırma:")
print(f"  TQuAD dev (Cell 4) - savasy baseline : F1 81.08, EM 62.78")
print(f"  TQuAD dev (Day 2)  - ft_model        : F1 73.33, EM 52.13")
print(f"  KAP eval  (BUGÜN)  - ft_model        : F1 {overall_results['f1']:.2f}, EM {overall_results['exact_match']:.2f}")

KAP eval: 60 soru

Tokenizing KAP eval (sliding window)...


Tokenizing KAP:   0%|          | 0/60 [00:00<?, ? examples/s]

KAP features: 13324 (expansion: 222.1x)

Feature dağılımı:
  ASELS: 2193 features / 12 soru = 183x
  BIMAS: 7030 features / 12 soru = 586x
  CWENE: 1062 features / 12 soru = 88x
  KTLEV: 1278 features / 12 soru = 106x
  YEOTK: 1761 features / 12 soru = 147x

Running ft_model inference on KAP features...


Inference:   0%|          | 0/417 [00:00<?, ?it/s]

Logits: start (13324, 384), end (13324, 384)

Postprocessing logits → text predictions...

KAP EVAL — OVERALL RESULTS (ft_model on KAP custom eval)
Exact Match : 30.00
F1          : 46.61

Karşılaştırma:
  TQuAD dev (Cell 4) - savasy baseline : F1 81.08, EM 62.78
  TQuAD dev (Day 2)  - ft_model        : F1 73.33, EM 52.13
  KAP eval  (BUGÜN)  - ft_model        : F1 46.61, EM 30.00


In [65]:
# === Cell KAP-BREAKDOWN: Şirket + soru tipi bazında F1/EM ===
from collections import defaultdict

# Manual labeling — taksonomide hangi soru hangi tipte
# (id'leri sırayla yazıyorum, daha önce konuştuğumuz dağılıma göre)

QUESTION_TYPES = {
    # ASELS (1-12)
    "asels_001": "factoid",  "asels_002": "numerik",  "asels_003": "factoid",
    "asels_004": "factoid",  "asels_005": "factoid",  "asels_006": "factoid",
    "asels_007": "numerik",  "asels_008": "numerik",  "asels_009": "numerik",
    "asels_010": "numerik",  "asels_011": "list",     "asels_012": "numerik",
    # BIMAS (1-12)
    "bimas_001": "factoid",  "bimas_002": "list",     "bimas_003": "numerik",
    "bimas_004": "numerik",  "bimas_005": "factoid",  "bimas_006": "numerik",
    "bimas_007": "numerik",  "bimas_008": "numerik",  "bimas_009": "factoid",
    "bimas_010": "numerik",  "bimas_011": "factoid",  "bimas_012": "list",
    # CWENE (1-12)
    "cwene_01": "factoid",   "cwene_02": "factoid",   "cwene_03": "factoid",
    "cwene_04": "factoid",   "cwene_05": "numerik",   "cwene_06": "numerik",
    "cwene_07": "numerik",   "cwene_08": "numerik",   "cwene_09": "numerik",
    "cwene_10": "list",      "cwene_11": "list",      "cwene_12": "neden-sonuc",
    # KTLEV (1-12)
    "ktlev_001": "factoid",  "ktlev_002": "factoid",  "ktlev_003": "factoid",
    "ktlev_004": "factoid",  "ktlev_005": "numerik",  "ktlev_006": "numerik",
    "ktlev_007": "numerik",  "ktlev_008": "numerik",  "ktlev_009": "numerik",
    "ktlev_010": "numerik",  "ktlev_011": "list",     "ktlev_012": "neden-sonuc",
    # YEOTK (1-12)
    "yeotk_001": "factoid",  "yeotk_002": "definitional", "yeotk_003": "numerik",
    "yeotk_004": "factoid",  "yeotk_005": "factoid",  "yeotk_006": "neden-sonuc",
    "yeotk_007": "factoid",  "yeotk_008": "numerik",  "yeotk_009": "numerik",
    "yeotk_010": "list",     "yeotk_011": "numerik",  "yeotk_012": "numerik",
}

# Her soru için ayrı F1/EM hesapla (squad_metric tek-soru bazında çalışır)
def per_question_metric(pred, gold):
    """squad_metric formatında tek soru için F1/EM."""
    preds = [{"id": "x", "prediction_text": pred}]
    refs = [{"id": "x", "answers": {"text": [gold], "answer_start": [0]}}]
    r = squad_metric.compute(predictions=preds, references=refs)
    return r["f1"], r["exact_match"]

# Tüm sorulara per-question score
per_question = {}
for ex in kap_examples:
    qid = ex["id"]
    gold = ex["answers"][0]["text"]
    pred = kap_predictions[qid]
    f1, em = per_question_metric(pred, gold)
    per_question[qid] = {"f1": f1, "em": em, "pred": pred, "gold": gold,
                         "question": ex["question"]}

# --- 1) ŞİRKET BAZINDA ---
print("="*70)
print("ŞİRKET BAZINDA F1/EM")
print("="*70)
by_company = defaultdict(list)
for qid, m in per_question.items():
    ticker = qid.split('_')[0]
    by_company[ticker].append(m)

print(f"{'Şirket':<10} {'F1':>8} {'EM':>8} {'Soru':>6}")
print("-"*40)
for ticker in ['asels', 'bimas', 'cwene', 'ktlev', 'yeotk']:
    items = by_company[ticker]
    avg_f1 = sum(m['f1'] for m in items) / len(items)
    avg_em = sum(m['em'] for m in items) / len(items)
    print(f"{ticker.upper():<10} {avg_f1:>7.2f}  {avg_em:>7.2f}  {len(items):>6}")

# --- 2) SORU TİPİ BAZINDA ---
print("\n" + "="*70)
print("SORU TİPİ BAZINDA F1/EM")
print("="*70)
by_type = defaultdict(list)
for qid, m in per_question.items():
    qtype = QUESTION_TYPES.get(qid, "?")
    by_type[qtype].append(m)

print(f"{'Tip':<15} {'F1':>8} {'EM':>8} {'Soru':>6}")
print("-"*45)
for qtype in ['factoid', 'numerik', 'list', 'neden-sonuc', 'definitional']:
    items = by_type.get(qtype, [])
    if not items:
        continue
    avg_f1 = sum(m['f1'] for m in items) / len(items)
    avg_em = sum(m['em'] for m in items) / len(items)
    print(f"{qtype:<15} {avg_f1:>7.2f}  {avg_em:>7.2f}  {len(items):>6}")

# --- 3) EN İYİ 5 + EN KÖTÜ 5 ---
print("\n" + "="*70)
print("EN İYİ 5 (F1 ile sıralı)")
print("="*70)
sorted_q = sorted(per_question.items(), key=lambda x: -x[1]['f1'])
for qid, m in sorted_q[:5]:
    qtype = QUESTION_TYPES.get(qid, "?")
    print(f"\n[{qid} | {qtype} | F1={m['f1']:.1f} EM={m['em']:.0f}]")
    print(f"  Q   : {m['question'][:80]}")
    print(f"  GOLD: '{m['gold'][:80]}'")
    print(f"  PRED: '{m['pred'][:80]}'")

print("\n" + "="*70)
print("EN KÖTÜ 10 (F1 ile sıralı)")
print("="*70)
for qid, m in sorted_q[-10:]:
    qtype = QUESTION_TYPES.get(qid, "?")
    print(f"\n[{qid} | {qtype} | F1={m['f1']:.1f} EM={m['em']:.0f}]")
    print(f"  Q   : {m['question'][:80]}")
    print(f"  GOLD: '{m['gold'][:80]}'")
    print(f"  PRED: '{m['pred'][:80]}'")

ŞİRKET BAZINDA F1/EM
Şirket           F1       EM   Soru
----------------------------------------
ASELS        21.40    16.67      12
BIMAS        61.67    50.00      12
CWENE        53.91    33.33      12
KTLEV        54.37    33.33      12
YEOTK        41.73    16.67      12

SORU TİPİ BAZINDA F1/EM
Tip                   F1       EM   Soru
---------------------------------------------
factoid           57.74    38.10      21
numerik           47.50    35.71      28
list              24.40     0.00       7
neden-sonuc       27.74     0.00       3
definitional       0.00     0.00       1

EN İYİ 5 (F1 ile sıralı)

[asels_005 | factoid | F1=100.0 EM=100]
  Q   : ASELSAN'ın Genel Müdürü kimdir?
  GOLD: 'Ahmet AKYOL'
  PRED: 'Ahmet AKYOL'

[asels_009 | numerik | F1=100.0 EM=100]
  Q   : ASELSAN'ın 31 Aralık 2025 itibarıyla ortalama personel sayısı kaçtır?
  GOLD: '14.143'
  PRED: '14.143'

[bimas_001 | factoid | F1=100.0 EM=100]
  Q   : Fas'taki ilk BİM mağazası hangi şehirde açılmıştır?


In [67]:
import json

# Tüm sonuçları tek dict'te topla
final_results = {
    "overall": {
        "f1": float(overall_results["f1"]),
        "exact_match": float(overall_results["exact_match"]),
        "n_questions": len(per_question),
    },
    "by_company": {
        ticker: {
            "f1": sum(m['f1'] for m in items) / len(items),
            "em": sum(m['em'] for m in items) / len(items),
            "n": len(items),
        }
        for ticker, items in by_company.items()
    },
    "by_question_type": {
        qtype: {
            "f1": sum(m['f1'] for m in items) / len(items),
            "em": sum(m['em'] for m in items) / len(items),
            "n": len(items),
        }
        for qtype, items in by_type.items()
    },
    "per_question": {
        qid: {
            "f1": m["f1"],
            "em": m["em"],
            "gold": m["gold"],
            "pred": m["pred"],
            "question": m["question"],
            "type": QUESTION_TYPES.get(qid, "?"),
            "company": qid.split('_')[0],
        }
        for qid, m in per_question.items()
    },
    "comparison": {
        "tquad_savasy_baseline": {"f1": 81.08, "em": 62.78},
        "tquad_ft_model": {"f1": 73.33, "em": 52.13},
        "kap_ft_model": {"f1": float(overall_results["f1"]), "em": float(overall_results["exact_match"])},
    }
}

with open('/kaggle/working/kap_eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"✅ Sonuçlar kaydedildi: /kaggle/working/kap_eval_results.json")

# Lokal'e indirebilmen için CSV de yapalım
import csv
with open('/kaggle/working/kap_eval_per_question.csv', 'w', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'company', 'type', 'question', 'gold', 'pred', 'f1', 'em'])
    for qid, m in per_question.items():
        writer.writerow([
            qid,
            qid.split('_')[0].upper(),
            QUESTION_TYPES.get(qid, "?"),
            m['question'],
            m['gold'].replace('\n', ' '),
            m['pred'].replace('\n', ' '),
            f"{m['f1']:.2f}",
            f"{m['em']:.0f}",
        ])
print("✅ CSV kaydedildi: /kaggle/working/kap_eval_per_question.csv")

✅ Sonuçlar kaydedildi: /kaggle/working/kap_eval_results.json
✅ CSV kaydedildi: /kaggle/working/kap_eval_per_question.csv
